<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-absolute-lumbar-window-scorer/67B2_postE50_absolute_lumbar_window_scorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 67B2 · Post-E50 — Supervised Absolute Lumbar Window Scorer

**Roadmap:** 67A relative localization → 67B absolute anchor audit → 67B1 orientation + geometry-only window audit → **67B2 (este notebook)** → locked validation → 67C axial pairing.

## Objetivo

Entrenar un **scorer supervisado pequeño e interpretable** que rankee todas las ventanas contiguas de 5 discos producidas por el runtime sagital congelado y seleccione la ventana lumbar absoluta más compatible con `L1-L2 ... L5-S1`.

El modelo **NO** recibe GT como feature. GT de RSNA se usa **únicamente dentro del split TRAIN** para construir el target de entrenamiento y evaluar un sub-split interno de desarrollo.

### Contrato metodológico

- Frozen sagittal checkpoint SHA-256:
  `cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`
- Outer split reproducible:
  `train=1382 / validation=296 / internal_test=297`
- **67B2 usa exclusivamente outer TRAIN.**
- `validation` externo: **bloqueado**.
- `internal_test`: **bloqueado**.
- test oficial RSNA: **bloqueado**.
- No se modifica `main`, Backend, Frontend ni checkpoints anteriores.
- No se cambia el modelo sagital congelado.
- No se entrena orientación.
- La regla DICOM de 67B1 fija `SUPERIOR_TO_INFERIOR`.
- Un GT TRAIN no monotónico según los 5 `ABSOLUTE_LEVEL_REFERENCE_POINT` queda excluido del target supervisado y se documenta; **no se repara ni relabela**.
- Estudios con menos de 5 instancias predichas no generan ventanas.
- Estudios con exactamente 5 instancias tienen una sola ventana y no aportan gradiente de ranking; se auditan pero no entran al entrenamiento.
- El scorer se entrena sólo con estudios con **2 o más ventanas candidatas**.
- La política final de abstención **NO se congela automáticamente** en este notebook. Se guarda una curva selectiva de DEV para decidirla antes de abrir validation.

## Runtime recomendado

**Google Colab L4 GPU.**

La GPU acelera la extracción de features porque vuelve a ejecutar el modelo sagital frozen sobre los estudios TRAIN. El MLP de ventanas es pequeño y no necesita una GPU potente por sí mismo.

El notebook tiene cache incremental en Google Drive. Si la sesión se corta, volver a ejecutar reanuda desde estudios ya procesados.

In [1]:
# ============================================================
# 0 — Environment / deterministic configuration
# ============================================================

import copy
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PFI_DRIVE_ROOT = Path(
    os.environ.get(
        "PFI_DRIVE_ROOT",
        "/content/drive/MyDrive/PFI_MVP" if IN_COLAB else "./PFI_MVP",
    )
)

RSNA_ROOT = Path(
    os.environ.get(
        "PFI_RSNA_ROOT",
        str(PFI_DRIVE_ROOT / "data" / "RSNA_LUMBAR_DISC"),
    )
)

CHECKPOINT_PATH = Path(
    os.environ.get(
        "PFI_POST_E50_SAGITTAL_CHECKPOINT",
        str(
            PFI_DRIVE_ROOT
            / "models"
            / "final"
            / "sagittal_spider_multiclass_final_best_cf11dcc0.pt"
        ),
    )
)

RESULTS_DIR = PFI_DRIVE_ROOT / "results" / "post_e50" / "67B2"
METRICS_DIR = PFI_DRIVE_ROOT / "metrics" / "post_e50" / "67B2"
MODEL_DIR = PFI_DRIVE_ROOT / "models" / "research" / "67B2"

for p in (RESULTS_DIR, METRICS_DIR, MODEL_DIR):
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_CHECKPOINT_SHA256 = (
    "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"
)

EXPECTED_CSV_SHA256 = {
    "train.csv":
        "f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
    "train_label_coordinates.csv":
        "416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
    "train_series_descriptions.csv":
        "bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33",
}

RUN_LOCKED_VALIDATION = False
RSNA_INTERNAL_TEST_LOCKED = True
OFFICIAL_TEST_ACCESSED = False

RUN_SMOKE_PARITY = True
RUN_FEATURE_EXTRACTION = True
TRAIN_SCORER = True

# None => process all eligible outer-TRAIN studies.
MAX_STUDIES = None

print("PFI_DRIVE_ROOT:", PFI_DRIVE_ROOT)
print("RSNA_ROOT:", RSNA_ROOT)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("RUN_LOCKED_VALIDATION:", RUN_LOCKED_VALIDATION)
print("RSNA_INTERNAL_TEST_LOCKED:", RSNA_INTERNAL_TEST_LOCKED)
print("OFFICIAL_TEST_ACCESSED:", OFFICIAL_TEST_ACCESSED)

assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False

IN_COLAB: True
Mounted at /content/drive
PFI_DRIVE_ROOT: /content/drive/MyDrive/PFI_MVP
RSNA_ROOT: /content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC
CHECKPOINT_PATH: /content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best_cf11dcc0.pt
RUN_LOCKED_VALIDATION: False
RSNA_INTERNAL_TEST_LOCKED: True
OFFICIAL_TEST_ACCESSED: False


In [2]:
# ============================================================
# 1 — Dependencies
# ============================================================

def _importable(name):
    try:
        module = __import__(name)
        return getattr(module, "__version__", "unknown")
    except ImportError:
        return None

required = {
    "torch": _importable("torch"),
    "SimpleITK": _importable("SimpleITK"),
    "scipy": _importable("scipy"),
    "pydicom": _importable("pydicom"),
    "PIL": _importable("PIL"),
}

print(required)

missing = [k for k, v in required.items() if v is None]

if missing:
    packages = []
    if "SimpleITK" in missing:
        packages.append("SimpleITK")
    if "scipy" in missing:
        packages.append("scipy")
    if "pydicom" in missing:
        packages.append("pydicom")
    if "PIL" in missing:
        packages.append("Pillow")
    if packages:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", *packages]
        )

import torch
import torch.nn as nn
import SimpleITK as sitk
import scipy
import pydicom
from PIL import Image
from scipy.optimize import linear_sum_assignment
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("torch:", torch.__version__)
print("SimpleITK:", sitk.Version_VersionString())
print("scipy:", scipy.__version__)
print("pydicom:", pydicom.__version__)
print("DEVICE:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory GB:", round(props.total_memory / 1024**3, 2))
else:
    print("WARNING: CPU runtime. L4 is recommended for full feature extraction.")

{'torch': '2.11.0+cu128', 'SimpleITK': None, 'scipy': '1.16.3', 'pydicom': None, 'PIL': '11.3.0'}
torch: 2.11.0+cu128
SimpleITK: 2.5.6
scipy: 1.16.3
pydicom: 3.0.2
DEVICE: cuda
GPU: NVIDIA L4
GPU memory GB: 22.03


In [3]:
# ============================================================
# 2 — Dataset identity + outer split
# ============================================================

CANONICAL_LEVELS = ["L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1"]
PRIMARY_REFERENCE_CONDITION = "Spinal Canal Stenosis"
PRIMARY_REFERENCE_SERIES = "Sagittal T2/STIR"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def opaque_id(raw_value) -> str:
    return hashlib.sha256(str(raw_value).encode("utf-8")).hexdigest()[:12]

for filename, expected in EXPECTED_CSV_SHA256.items():
    path = RSNA_ROOT / filename
    assert path.is_file(), f"Missing {path}"
    actual = sha256_file(path)
    print(filename, actual, "MATCH" if actual == expected else "MISMATCH")
    assert actual == expected, f"{filename} SHA mismatch"

train_df = pd.read_csv(
    RSNA_ROOT / "train.csv",
    dtype={"study_id": "string"},
)

coords_df = pd.read_csv(
    RSNA_ROOT / "train_label_coordinates.csv",
    dtype={"study_id": "string", "series_id": "string"},
)

series_df = pd.read_csv(
    RSNA_ROOT / "train_series_descriptions.csv",
    dtype={"study_id": "string", "series_id": "string"},
)

print("shapes:", train_df.shape, coords_df.shape, series_df.shape)

assert train_df.shape == (1975, 26)
assert coords_df.shape == (48692, 7)
assert series_df.shape == (6294, 3)

def normalize_level(raw):
    if raw is None:
        return None
    try:
        if pd.isna(raw):
            return None
    except Exception:
        pass
    text = str(raw).strip().upper().replace("/", "-").replace("_", "-")
    return text if text in CANONICAL_LEVELS else None

coords_df = coords_df.copy()
coords_df["level_raw"] = coords_df["level"]
coords_df["level_normalized"] = coords_df["level_raw"].apply(normalize_level)

def build_reproducible_study_split(study_ids, seed=2026, train_frac=0.70, validation_frac=0.15):
    ordered = sorted(str(x) for x in study_ids)
    rng = np.random.default_rng(seed)
    shuffled = list(ordered)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * train_frac)
    n_val = int(n * validation_frac)
    return {
        "train": sorted(shuffled[:n_train]),
        "validation": sorted(shuffled[n_train:n_train+n_val]),
        "internal_test": sorted(shuffled[n_train+n_val:]),
    }

outer_split = build_reproducible_study_split(series_df["study_id"].dropna().unique())

print({k: len(v) for k, v in outer_split.items()})

assert len(outer_split["train"]) == 1382
assert len(outer_split["validation"]) == 296
assert len(outer_split["internal_test"]) == 297

assert set(outer_split["train"]).isdisjoint(outer_split["validation"])
assert set(outer_split["train"]).isdisjoint(outer_split["internal_test"])
assert set(outer_split["validation"]).isdisjoint(outer_split["internal_test"])

print("Outer split leakage audit: PASS")

train.csv f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528 MATCH
train_label_coordinates.csv 416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635 MATCH
train_series_descriptions.csv bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33 MATCH
shapes: (1975, 26) (48692, 7) (6294, 3)
{'train': 1382, 'validation': 296, 'internal_test': 297}
Outer split leakage audit: PASS


In [4]:
# ============================================================
# 3 — Primary reference table + eligible TRAIN pool
# ============================================================

primary_reference_df = coords_df.merge(
    series_df[
        ["study_id", "series_id", "series_description"]
    ].drop_duplicates(),
    on=["study_id", "series_id"],
    how="left",
    validate="many_to_one",
)

primary_reference_df = primary_reference_df[
    (
        primary_reference_df["condition"].astype(str).str.strip()
        == PRIMARY_REFERENCE_CONDITION
    )
    &
    (
        primary_reference_df["series_description"].astype(str).str.strip()
        == PRIMARY_REFERENCE_SERIES
    )
    &
    (
        primary_reference_df["level_normalized"].isin(CANONICAL_LEVELS)
    )
    &
    primary_reference_df["instance_number"].notna()
    &
    primary_reference_df["x"].notna()
    &
    primary_reference_df["y"].notna()
].copy()

print("primary rows:", len(primary_reference_df))
print("primary studies:", primary_reference_df["study_id"].nunique())
print("primary series:", primary_reference_df["series_id"].nunique())

assert len(primary_reference_df) == 9748
assert primary_reference_df["study_id"].nunique() == 1973
assert primary_reference_df["series_id"].nunique() == 1973

per_study_levels = (
    primary_reference_df
    .groupby("study_id")["level_normalized"]
    .agg(lambda s: set(s))
)

complete_reference_studies = sorted(
    sid
    for sid, levels in per_study_levels.items()
    if set(CANONICAL_LEVELS).issubset(levels)
)

outer_train_set = set(outer_split["train"])
outer_validation_set = set(outer_split["validation"])

eligible_train_raw = sorted(
    sid for sid in complete_reference_studies
    if sid in outer_train_set
)

eligible_validation_raw = sorted(
    sid for sid in complete_reference_studies
    if sid in outer_validation_set
)

print("Eligible TRAIN:", len(eligible_train_raw))
print("Eligible validation (only for frozen smoke parity selection):", len(eligible_validation_raw))

# Expected from 67B1 dataset/split contract.
assert len(eligible_train_raw) == 1322
assert len(eligible_validation_raw) == 290

if MAX_STUDIES is not None:
    eligible_train_raw = eligible_train_raw[:int(MAX_STUDIES)]
    print("DEBUG MAX_STUDIES applied:", len(eligible_train_raw))

primary rows: 9748
primary studies: 1973
primary series: 1973
Eligible TRAIN: 1322
Eligible validation (only for frozen smoke parity selection): 290


In [5]:
# ============================================================
# 4 — Frozen sagittal runtime
# ============================================================

def checkpoint_state_dict(checkpoint: Any) -> Mapping[str, torch.Tensor]:
    if isinstance(checkpoint, Mapping):
        for key in ("model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, Mapping):
                return normalize_state_dict(value)
        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return normalize_state_dict(checkpoint)
    raise ValueError("Checkpoint has no usable state_dict")

def normalize_state_dict(state_dict):
    out = {}
    for key, value in state_dict.items():
        clean = str(key)
        for prefix in ("module.", "model."):
            if clean.startswith(prefix):
                clean = clean[len(prefix):]
        out[clean] = value
    return out

class SagittalDoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class SagittalUNet2D(nn.Module):
    def __init__(self, in_channels=1, num_classes=4, base_channels=16):
        super().__init__()
        self.enc1 = SagittalDoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = SagittalDoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = SagittalDoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = SagittalDoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, 2)
        self.dec3 = SagittalDoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, 2)
        self.dec2 = SagittalDoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, 2)
        self.dec1 = SagittalDoubleConv(base_channels * 2, base_channels)
        self.out_conv = nn.Conv2d(base_channels, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)

def build_checkpoint_model(checkpoint):
    state = checkpoint_state_dict(checkpoint)
    base_channels = (
        int(checkpoint["base_channels"])
        if isinstance(checkpoint, Mapping) and checkpoint.get("base_channels") is not None
        else int(state["enc1.block.0.weight"].shape[0])
    )
    num_classes = (
        int(checkpoint["num_classes"])
        if isinstance(checkpoint, Mapping) and checkpoint.get("num_classes") is not None
        else int(state["out_conv.weight"].shape[0])
    )
    target_size = (256, 256)
    if isinstance(checkpoint, Mapping) and checkpoint.get("target_size") is not None:
        raw = checkpoint["target_size"]
        target_size = (int(raw[0]), int(raw[1]))

    model = SagittalUNet2D(
        num_classes=num_classes,
        base_channels=base_channels,
    )
    model.load_state_dict(state, strict=True)
    return model, {
        "base_channels": base_channels,
        "num_classes": num_classes,
        "target_size": target_size,
    }

assert CHECKPOINT_PATH.is_file(), f"Missing checkpoint: {CHECKPOINT_PATH}"
checkpoint_sha = sha256_file(CHECKPOINT_PATH)

print("checkpoint SHA:", checkpoint_sha)
assert checkpoint_sha == EXPECTED_CHECKPOINT_SHA256

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

sagittal_model, sagittal_meta = build_checkpoint_model(checkpoint)
sagittal_model.to(DEVICE)
sagittal_model.eval()

print("Frozen model loaded:", sagittal_meta)
print("Training mode:", sagittal_model.training)
assert sagittal_model.training is False

checkpoint SHA: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944
Frozen model loaded: {'base_channels': 16, 'num_classes': 4, 'target_size': (256, 256)}
Training mode: False


In [6]:
# ============================================================
# 5 — Geometry + preprocessing helpers
# ============================================================

def robust_percentile_normalize(array, p_low=1.0, p_high=99.0):
    value = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(value)
    if not finite.any():
        return np.zeros_like(value, dtype=np.float32)
    low, high = np.percentile(value[finite], [p_low, p_high])
    if float(high) <= float(low):
        return np.zeros_like(value, dtype=np.float32)
    clipped = np.clip(value, low, high)
    return ((clipped - low) / (float(high) - float(low) + 1e-8)).astype(np.float32)

def resize_image(array, target_size):
    normalized = robust_percentile_normalize(array)
    image = Image.fromarray(
        np.clip(normalized * 255.0, 0, 255).astype(np.uint8)
    )
    resized = image.resize(
        (target_size[1], target_size[0]),
        resample=Image.Resampling.BILINEAR,
    )
    return np.asarray(resized, dtype=np.float32) / 255.0

def connected_instances(binary, min_pixels=20):
    labelled = sitk.GetArrayFromImage(
        sitk.ConnectedComponent(
            sitk.GetImageFromArray(binary.astype(np.uint8))
        )
    )
    instances = []
    for value in sorted(int(x) for x in np.unique(labelled) if int(x) != 0):
        component = labelled == value
        if int(component.sum()) >= min_pixels:
            instances.append(component)
    instances.sort(key=lambda mask: float(np.where(mask)[0].mean()))
    return instances

def component_geometry(mask):
    idx = np.where(mask)
    return {
        "centroid_row": float(np.mean(idx[0])),
        "centroid_col": float(np.mean(idx[1])),
        "area_px": int(mask.sum()),
        "border_touch": bool(
            idx[0].min() == 0
            or idx[1].min() == 0
            or idx[0].max() == mask.shape[0] - 1
            or idx[1].max() == mask.shape[1] - 1
        ),
    }

def dicom_pixel_to_patient_xyz(
    column_index,
    row_index,
    pixel_spacing,
    image_position_patient,
    image_orientation_patient,
):
    row_cosines = np.asarray(image_orientation_patient[:3], dtype=np.float64)
    column_cosines = np.asarray(image_orientation_patient[3:], dtype=np.float64)
    origin = np.asarray(image_position_patient, dtype=np.float64)
    row_spacing, column_spacing = pixel_spacing
    return (
        origin
        + float(column_index) * column_spacing * row_cosines
        + float(row_index) * row_spacing * column_cosines
    )

def check_orientation_valid(iop):
    if len(iop) != 6:
        return False
    row = np.asarray(iop[:3], dtype=np.float64)
    col = np.asarray(iop[3:], dtype=np.float64)
    return bool(
        abs(np.linalg.norm(row) - 1.0) < 1e-2
        and abs(np.linalg.norm(col) - 1.0) < 1e-2
        and abs(float(np.dot(row, col))) < 1e-2
    )

def order_dicom_series_by_geometry(dicom_paths):
    records = []
    tags = [
        "Rows", "Columns", "PixelSpacing",
        "ImagePositionPatient", "ImageOrientationPatient",
        "InstanceNumber", "AnatomicalOrientationType",
    ]
    for p in dicom_paths:
        ds = pydicom.dcmread(
            str(p),
            stop_before_pixels=True,
            specific_tags=tags,
        )
        required = [
            "Rows", "Columns", "PixelSpacing",
            "ImagePositionPatient", "ImageOrientationPatient",
            "InstanceNumber",
        ]
        if not all(hasattr(ds, tag) for tag in required):
            continue
        iop = tuple(float(v) for v in ds.ImageOrientationPatient)
        ipp = tuple(float(v) for v in ds.ImagePositionPatient)
        row = np.asarray(iop[:3], dtype=np.float64)
        col = np.asarray(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        position_mm = float(np.dot(np.asarray(ipp), normal))
        records.append({
            "path": p,
            "position_mm": position_mm,
            "instance_number": int(ds.InstanceNumber),
            "ipp": ipp,
            "iop": iop,
            "pixel_spacing": tuple(float(v) for v in ds.PixelSpacing),
            "rows": int(ds.Rows),
            "columns": int(ds.Columns),
            "anatomical_orientation_type":
                getattr(ds, "AnatomicalOrientationType", None),
        })
    records.sort(key=lambda r: r["position_mm"])
    return records

def consensus_union_find(centroids_xyz, distance_threshold_mm=15.0):
    n = len(centroids_xyz)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if np.linalg.norm(
                np.asarray(centroids_xyz[i]) - np.asarray(centroids_xyz[j])
            ) <= distance_threshold_mm:
                union(i, j)

    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)

    return [v for _, v in sorted(groups.items())]

def spine_axis_from_points(points_xyz):
    centered = points_xyz - points_xyz.mean(axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    axis = vt[0]
    return axis / np.linalg.norm(axis)

def project_points_onto_axis(points_xyz, axis, reference_point):
    axis = np.asarray(axis, dtype=np.float64)
    ref = np.asarray(reference_point, dtype=np.float64)
    return [
        float(np.dot(np.asarray(p, dtype=np.float64) - ref, axis))
        for p in points_xyz
    ]

def compute_instance_confidence(
    supporting_slice_count,
    centroid_spread_mm,
    mean_segmentation_confidence,
):
    multi_slice_score = min(supporting_slice_count / 3.0, 1.0)
    stability_score = float(
        np.clip(1.0 - centroid_spread_mm / 20.0, 0.0, 1.0)
    )
    return round(
        0.40 * multi_slice_score
        + 0.30 * stability_score
        + 0.30 * mean_segmentation_confidence,
        4,
    )

def axis_hungarian_assignment_no_threshold(ref_positions, pred_positions):
    if not ref_positions or not pred_positions:
        return []
    matrix = np.abs(
        np.asarray(ref_positions, dtype=np.float64)[:, None]
        - np.asarray(pred_positions, dtype=np.float64)[None, :]
    )
    rows, cols = linear_sum_assignment(matrix)
    return [
        {
            "ref_index": int(r),
            "pred_index": int(c),
            "distance_mm": float(matrix[r, c]),
        }
        for r, c in zip(rows, cols)
    ]

print("Geometry / preprocessing helpers: READY")

Geometry / preprocessing helpers: READY


In [7]:
# ============================================================
# 6 — 67B1 orientation rule (GT-free)
# ============================================================

REFERENCE_SUPERIOR_DIRECTION = np.array([0.0, 0.0, 1.0])
NUMERICAL_TOLERANCE = 1e-6

def classify_anatomical_orientation_type(raw):
    if raw is None or str(raw).strip() == "":
        return {"effective": "BIPED_DEFAULT", "supported": True}
    normalized = str(raw).strip().upper()
    if normalized == "BIPED":
        return {"effective": "BIPED_EXPLICIT", "supported": True}
    if normalized == "QUADRUPED":
        return {
            "effective": "QUADRUPED",
            "supported": False,
            "reason": "UNSUPPORTED_QUADRUPED_ORIENTATION",
        }
    return {
        "effective": f"UNRECOGNIZED:{normalized}",
        "supported": False,
        "reason": "UNRECOGNIZED_ANATOMICAL_ORIENTATION_TYPE",
    }

def resolve_axis_direction(predicted_axis, anatomical_orientation_type=None):
    if predicted_axis is None:
        return {
            "status": "ABSTAIN",
            "oriented_axis": None,
            "reason": "NO_PREDICTED_AXIS",
        }

    cls = classify_anatomical_orientation_type(
        anatomical_orientation_type
    )

    if not cls["supported"]:
        return {
            "status": "ABSTAIN",
            "oriented_axis": None,
            "reason": cls["reason"],
        }

    raw_axis = np.asarray(predicted_axis, dtype=np.float64)
    alignment = float(np.dot(raw_axis, REFERENCE_SUPERIOR_DIRECTION))

    if abs(alignment) <= NUMERICAL_TOLERANCE:
        return {
            "status": "ABSTAIN",
            "oriented_axis": None,
            "reason": "AXIS_ORTHOGONAL_TO_PATIENT_Z",
            "axis_alignment": alignment,
        }

    oriented_axis = -raw_axis if alignment > 0 else raw_axis

    return {
        "status": "RESOLVED",
        "oriented_axis": oriented_axis,
        "axis_alignment": alignment,
        "effective_anatomical_orientation_type": cls["effective"],
        "canonical_axis_semantics": "SUPERIOR_TO_INFERIOR",
    }

assert np.allclose(
    resolve_axis_direction(
        np.array([0.0, 0.0, 1.0]),
        None,
    )["oriented_axis"],
    [0.0, 0.0, -1.0],
)

print("67B1 orientation rule: READY")

67B1 orientation rule: READY


In [8]:
# ============================================================
# 7 — Frozen inference per study
# ============================================================

TRAIN_IMAGES_ROOT = RSNA_ROOT / "train_images"

def run_frozen_sagittal_on_study(study_id_raw):
    study_rows = series_df[
        series_df["study_id"].astype(str) == str(study_id_raw)
    ]

    sag_rows = study_rows[
        study_rows["series_description"].astype(str).str.strip()
        == PRIMARY_REFERENCE_SERIES
    ]

    if len(sag_rows) == 0:
        return {"status": "NO_SAGITTAL_T2_STIR"}

    series_id_raw = str(sag_rows.iloc[0]["series_id"])
    series_dir = (
        TRAIN_IMAGES_ROOT
        / str(study_id_raw)
        / series_id_raw
    )

    dicom_paths = sorted(series_dir.glob("*.dcm"))
    if not dicom_paths:
        return {"status": "NO_DICOM_FILES"}

    slices = order_dicom_series_by_geometry(dicom_paths)
    if len(slices) < 2:
        return {"status": "INSUFFICIENT_SLICES"}

    anatomical_orientation_type = next(
        (
            s["anatomical_orientation_type"]
            for s in slices
            if s.get("anatomical_orientation_type")
        ),
        None,
    )

    target_size = tuple(sagittal_meta["target_size"])
    predicted_components = []

    for slice_index, rec in enumerate(slices):
        ds = pydicom.dcmread(str(rec["path"]))
        native = ds.pixel_array.astype(np.float32)

        prepared = resize_image(native, target_size)
        tensor = (
            torch.from_numpy(prepared[None, None])
            .float()
            .to(DEVICE, non_blocking=True)
        )

        with torch.inference_mode():
            if DEVICE.type == "cuda":
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    logits = sagittal_model(tensor)
            else:
                logits = sagittal_model(tensor)

            probabilities = torch.softmax(logits.float(), dim=1)[0]
            prediction = (
                torch.argmax(probabilities, dim=0)
                .cpu()
                .numpy()
                .astype(np.uint8)
            )
            confidence_map = (
                torch.max(probabilities, dim=0)
                .values
                .cpu()
                .numpy()
                .astype(np.float32)
            )

        disc_mask = prediction == 3

        for component in connected_instances(disc_mask):
            geom = component_geometry(component)

            row_native = (
                geom["centroid_row"]
                * native.shape[0]
                / target_size[0]
            )
            col_native = (
                geom["centroid_col"]
                * native.shape[1]
                / target_size[1]
            )

            if not (
                0 <= row_native < rec["rows"]
                and 0 <= col_native < rec["columns"]
                and check_orientation_valid(rec["iop"])
            ):
                continue

            xyz = dicom_pixel_to_patient_xyz(
                col_native,
                row_native,
                rec["pixel_spacing"],
                rec["ipp"],
                rec["iop"],
            )

            if not np.all(np.isfinite(xyz)):
                continue

            predicted_components.append({
                "slice_index": slice_index,
                "centroid_xyz": xyz,
                "area_px": geom["area_px"],
                "border_touch": geom["border_touch"],
                # Preserve frozen 67B/67B1 semantics.
                "mean_confidence": float(confidence_map.mean()),
            })

    if not predicted_components:
        return {"status": "NO_DISC_COMPONENTS"}

    groups = consensus_union_find(
        [c["centroid_xyz"] for c in predicted_components],
        distance_threshold_mm=15.0,
    )

    consensus_instances = []

    for indices in groups:
        members = [predicted_components[i] for i in indices]
        centroids = np.stack([m["centroid_xyz"] for m in members])
        mean_centroid = centroids.mean(axis=0)

        spread = (
            float(
                np.max(
                    np.linalg.norm(
                        centroids - mean_centroid,
                        axis=1,
                    )
                )
            )
            if len(members) > 1
            else 0.0
        )

        mean_seg_conf = float(
            np.mean([m["mean_confidence"] for m in members])
        )

        consensus_instances.append({
            "centroid_xyz": mean_centroid,
            "supporting_slice_count": len(members),
            "instance_confidence":
                compute_instance_confidence(
                    len(members),
                    spread,
                    mean_seg_conf,
                ),
            "centroid_spread_mm": spread,
            "mean_area_px":
                float(np.mean([m["area_px"] for m in members])),
            "border_touch":
                bool(any(m["border_touch"] for m in members)),
        })

    if len(consensus_instances) < 2:
        return {
            "status": "INSUFFICIENT_CONSENSUS_INSTANCES",
            "predicted_instance_count": len(consensus_instances),
        }

    axis = spine_axis_from_points(
        np.stack([x["centroid_xyz"] for x in consensus_instances])
    )
    ref = np.asarray(consensus_instances[0]["centroid_xyz"])

    raw_positions = project_points_onto_axis(
        [x["centroid_xyz"] for x in consensus_instances],
        axis,
        ref,
    )

    for inst, pos in zip(consensus_instances, raw_positions):
        inst["position_along_axis_mm"] = pos

    consensus_instances.sort(
        key=lambda x: x["position_along_axis_mm"]
    )

    return {
        "status": "EXECUTED",
        "series_id_opaque": opaque_id(series_id_raw),
        "predicted_instance_count": len(consensus_instances),
        "consensus_instances": consensus_instances,
        "predicted_axis": axis,
        "predicted_axis_reference_point": ref,
        "anatomical_orientation_type": anatomical_orientation_type,
        "n_slices": len(slices),
    }

print("Frozen study inference: READY")

Frozen study inference: READY


In [9]:
# ============================================================
# 8 — Frozen smoke parity hard-stop
# ============================================================

EXPECTED_67B_SMOKE_PARITY = {
    "4f06df2fd53b": 8,
    "d41a396f20c6": 9,
    "ef2ff5b618cf": 7,
}

smoke_raw_by_opaque = {
    opaque_id(sid): sid
    for sid in eligible_validation_raw
}

missing_smoke = [
    opaque
    for opaque in EXPECTED_67B_SMOKE_PARITY
    if opaque not in smoke_raw_by_opaque
]

assert not missing_smoke, (
    f"Could not resolve frozen smoke IDs: {missing_smoke}"
)

smoke_parity_rows = []

if RUN_SMOKE_PARITY:
    for opaque, expected_count in EXPECTED_67B_SMOKE_PARITY.items():
        sid = smoke_raw_by_opaque[opaque]
        result = run_frozen_sagittal_on_study(sid)

        observed = (
            result.get("predicted_instance_count")
            if result.get("status") == "EXECUTED"
            else None
        )

        ok = observed == expected_count

        row = {
            "study_id_opaque": opaque,
            "expected_count": expected_count,
            "observed_count": observed,
            "status": result.get("status"),
            "parity": "PASS" if ok else "FAIL",
        }

        smoke_parity_rows.append(row)
        print(row)

    assert all(
        x["parity"] == "PASS"
        for x in smoke_parity_rows
    ), "HARD STOP: frozen smoke parity failed"

    print("SMOKE PARITY: PASS")
else:
    print("RUN_SMOKE_PARITY=False")

{'study_id_opaque': '4f06df2fd53b', 'expected_count': 8, 'observed_count': 8, 'status': 'EXECUTED', 'parity': 'PASS'}
{'study_id_opaque': 'd41a396f20c6', 'expected_count': 9, 'observed_count': 9, 'status': 'EXECUTED', 'parity': 'PASS'}
{'study_id_opaque': 'ef2ff5b618cf', 'expected_count': 7, 'observed_count': 7, 'status': 'EXECUTED', 'parity': 'PASS'}
SMOKE PARITY: PASS


In [10]:
# ============================================================
# 9 — GT resolver, windows, target QC, inference-only features
# ============================================================

WINDOW_SIZE = 5

def resolve_gt_level_xyz_for_study(study_id_raw):
    rows = primary_reference_df[
        primary_reference_df["study_id"].astype(str)
        == str(study_id_raw)
    ]

    result = {}

    for _, row in rows.iterrows():
        level = normalize_level(row["level_raw"])
        if level not in CANONICAL_LEVELS:
            continue

        series_id = str(row["series_id"])
        instance_number = int(row["instance_number"])

        dcm_path = (
            TRAIN_IMAGES_ROOT
            / str(study_id_raw)
            / series_id
            / f"{instance_number}.dcm"
        )

        if not dcm_path.is_file():
            matches = list(
                (
                    TRAIN_IMAGES_ROOT
                    / str(study_id_raw)
                    / series_id
                ).glob(f"*{instance_number}*.dcm")
            )
            dcm_path = matches[0] if matches else None

        if dcm_path is None or not Path(dcm_path).is_file():
            continue

        ds = pydicom.dcmread(
            str(dcm_path),
            stop_before_pixels=True,
        )

        required = [
            "PixelSpacing",
            "ImagePositionPatient",
            "ImageOrientationPatient",
        ]

        if not all(hasattr(ds, tag) for tag in required):
            continue

        spacing = tuple(float(v) for v in ds.PixelSpacing)
        ipp = tuple(float(v) for v in ds.ImagePositionPatient)
        iop = tuple(float(v) for v in ds.ImageOrientationPatient)

        if not check_orientation_valid(iop):
            continue

        xyz = dicom_pixel_to_patient_xyz(
            float(row["x"]),
            float(row["y"]),
            spacing,
            ipp,
            iop,
        )

        if np.all(np.isfinite(xyz)):
            result[level] = xyz

    return result

def enumerate_candidate_windows(n_instances):
    if n_instances < WINDOW_SIZE:
        return []
    return [
        {
            "window_index": i,
            "rank_start": i + 1,
            "rank_end": i + WINDOW_SIZE,
            "ranks": list(
                range(i + 1, i + WINDOW_SIZE + 1)
            ),
        }
        for i in range(n_instances - WINDOW_SIZE + 1)
    ]

def orient_and_sort_instances(result):
    direction = resolve_axis_direction(
        result.get("predicted_axis"),
        result.get("anatomical_orientation_type"),
    )

    if direction["status"] != "RESOLVED":
        return direction, None

    instances = copy.deepcopy(
        result["consensus_instances"]
    )

    axis = np.asarray(
        direction["oriented_axis"],
        dtype=np.float64,
    )

    ref = np.asarray(
        result["predicted_axis_reference_point"],
        dtype=np.float64,
    )

    positions = project_points_onto_axis(
        [x["centroid_xyz"] for x in instances],
        axis,
        ref,
    )

    for inst, pos in zip(instances, positions):
        inst["position_along_axis_mm"] = float(pos)

    instances.sort(
        key=lambda x: x["position_along_axis_mm"]
    )

    return direction, instances

def gt_target_qc_and_best_window(
    candidate_windows,
    ordered_instances,
    oriented_axis,
    reference_point,
    gt_level_xyz,
):
    if len(gt_level_xyz) != 5:
        return {
            "status": "GT_INCOMPLETE",
            "target_window_index": None,
        }

    gt_positions = project_points_onto_axis(
        [gt_level_xyz[lv] for lv in CANONICAL_LEVELS],
        oriented_axis,
        reference_point,
    )

    # Target quality control: canonical GT points must be
    # strictly increasing along the already-GT-free oriented axis.
    monotonic = all(
        gt_positions[i] < gt_positions[i + 1]
        for i in range(4)
    )

    if not monotonic:
        return {
            "status": "GT_NON_MONOTONIC_REFERENCE",
            "target_window_index": None,
            "gt_positions_mm": gt_positions,
        }

    pred_positions = [
        x["position_along_axis_mm"]
        for x in ordered_instances
    ]

    costs = []

    for w in candidate_windows:
        window_positions = [
            pred_positions[r - 1]
            for r in w["ranks"]
        ]

        assignment = axis_hungarian_assignment_no_threshold(
            gt_positions,
            window_positions,
        )

        cost = (
            float(sum(a["distance_mm"] for a in assignment))
            if len(assignment) == 5
            else float("inf")
        )

        costs.append(
            (w["window_index"], cost)
        )

    costs.sort(key=lambda x: x[1])

    if not costs or not np.isfinite(costs[0][1]):
        return {
            "status": "GT_TARGET_UNAVAILABLE",
            "target_window_index": None,
        }

    # Exact numerical tie only; no anatomical threshold invented.
    if (
        len(costs) > 1
        and abs(costs[0][1] - costs[1][1]) <= 1e-9
    ):
        return {
            "status": "GT_TARGET_TIE",
            "target_window_index": None,
            "ranked_gt_costs": costs,
        }

    return {
        "status": "TARGET_OK",
        "target_window_index": int(costs[0][0]),
        "best_gt_cost_mm": float(costs[0][1]),
        "second_gt_cost_mm":
            float(costs[1][1]) if len(costs) > 1 else None,
        "gt_positions_mm": gt_positions,
    }

def _safe_stats(values):
    arr = np.asarray(values, dtype=np.float64)
    return {
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
    }

def build_window_feature_row(
    window,
    ordered_instances,
    oriented_axis,
    reference_point,
):
    n = len(ordered_instances)
    n_windows = n - WINDOW_SIZE + 1
    idx = int(window["window_index"])

    members = [
        ordered_instances[r - 1]
        for r in window["ranks"]
    ]

    positions = np.asarray(
        [
            x["position_along_axis_mm"]
            for x in ordered_instances
        ],
        dtype=np.float64,
    )

    member_positions = np.asarray(
        [
            x["position_along_axis_mm"]
            for x in members
        ],
        dtype=np.float64,
    )

    gaps = np.diff(member_positions)
    gap_mean = float(np.mean(gaps))
    gap_std = float(np.std(gaps))
    gap_cv = (
        float(gap_std / abs(gap_mean))
        if gap_mean != 0
        else 0.0
    )

    left_margin = int(window["rank_start"] - 1)
    right_margin = int(n - window["rank_end"])

    left_gap_available = left_margin > 0
    right_gap_available = right_margin > 0

    left_gap = (
        float(
            member_positions[0]
            - positions[window["rank_start"] - 2]
        )
        if left_gap_available
        else 0.0
    )

    right_gap = (
        float(
            positions[window["rank_end"]]
            - member_positions[-1]
        )
        if right_gap_available
        else 0.0
    )

    conf = _safe_stats(
        [x["instance_confidence"] for x in members]
    )
    support = _safe_stats(
        [x["supporting_slice_count"] for x in members]
    )
    spread = _safe_stats(
        [x["centroid_spread_mm"] for x in members]
    )
    area = _safe_stats(
        [x["mean_area_px"] for x in members]
    )

    axis = np.asarray(oriented_axis, dtype=np.float64)
    ref = np.asarray(reference_point, dtype=np.float64)

    residuals = []
    for x in members:
        delta = np.asarray(x["centroid_xyz"], dtype=np.float64) - ref
        longitudinal = float(np.dot(delta, axis))
        transverse = delta - longitudinal * axis
        residuals.append(float(np.linalg.norm(transverse)))

    residual = _safe_stats(residuals)

    global_span = float(positions[-1] - positions[0])
    window_center = float(
        0.5 * (member_positions[0] + member_positions[-1])
    )
    global_center = float(
        0.5 * (positions[0] + positions[-1])
    )

    return {
        "n_instances": float(n),
        "n_candidate_windows": float(n_windows),

        "window_index_norm":
            float(idx / max(n_windows - 1, 1)),
        "rank_start_norm":
            float((window["rank_start"] - 1) / max(n - 1, 1)),
        "rank_end_norm":
            float((window["rank_end"] - 1) / max(n - 1, 1)),

        "left_margin_count": float(left_margin),
        "right_margin_count": float(right_margin),
        "min_margin_count": float(min(left_margin, right_margin)),
        "margin_balance_abs":
            float(abs(left_margin - right_margin)),

        "window_span_mm":
            float(member_positions[-1] - member_positions[0]),
        "global_span_mm":
            global_span,
        "window_center_offset_from_global_center_mm":
            float(window_center - global_center),

        "gap_mean_mm": gap_mean,
        "gap_std_mm": gap_std,
        "gap_cv": gap_cv,
        "gap_min_mm": float(np.min(gaps)),
        "gap_max_mm": float(np.max(gaps)),
        "gap_range_mm":
            float(np.max(gaps) - np.min(gaps)),

        "gap_1_mm": float(gaps[0]),
        "gap_2_mm": float(gaps[1]),
        "gap_3_mm": float(gaps[2]),
        "gap_4_mm": float(gaps[3]),

        "left_outside_gap_available":
            float(left_gap_available),
        "right_outside_gap_available":
            float(right_gap_available),
        "left_outside_gap_mm": left_gap,
        "right_outside_gap_mm": right_gap,

        "left_boundary_gap_ratio":
            float(left_gap / gap_mean)
            if left_gap_available and gap_mean != 0
            else 0.0,
        "right_boundary_gap_ratio":
            float(right_gap / gap_mean)
            if right_gap_available and gap_mean != 0
            else 0.0,

        "confidence_mean": conf["mean"],
        "confidence_std": conf["std"],
        "confidence_min": conf["min"],
        "confidence_max": conf["max"],

        "support_mean": support["mean"],
        "support_std": support["std"],
        "support_min": support["min"],
        "support_max": support["max"],

        "spread_mean_mm": spread["mean"],
        "spread_std_mm": spread["std"],
        "spread_max_mm": spread["max"],

        "area_mean_px": area["mean"],
        "area_std_px": area["std"],
        "area_min_px": area["min"],
        "area_max_px": area["max"],
        "log1p_area_mean":
            float(np.log1p(max(area["mean"], 0.0))),

        "border_touch_count":
            float(sum(bool(x["border_touch"]) for x in members)),
        "border_touch_fraction":
            float(np.mean([bool(x["border_touch"]) for x in members])),

        "axis_residual_mean_mm": residual["mean"],
        "axis_residual_std_mm": residual["std"],
        "axis_residual_max_mm": residual["max"],
    }

print("Target + feature pipeline: READY")

Target + feature pipeline: READY


In [12]:
# ============================================================
# 67B2 — L4 OPTIMIZATION OVERRIDE
#
# Paste/run this cell BEFORE:
#   # 10 — Incremental TRAIN feature extraction with Drive cache
#
# Changes execution ONLY:
# - same frozen checkpoint
# - same preprocessing
# - same connected components
# - same 15 mm consensus
# - same features / GT target
#
# Optimizations:
# 1) batch multiple sagittal slices per GPU forward
# 2) stage ONE DICOM series at a time to local /content
# 3) preserve existing Drive feature cache
#
# HARD STOP:
# optimized implementation must reproduce frozen smoke counts
# 8 / 9 / 7 before continuing.
# ============================================================

import shutil
import time
from pathlib import Path

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INFERENCE_BATCH_SIZE = 32

# Recommended for Google Drive:
# copy one series to local SSD, process, then delete.
STAGE_SERIES_LOCALLY = True

LOCAL_STAGE_ROOT = Path(
    "/content/pfi_67b2_series_stage"
)

LOCAL_STAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "DEVICE:",
    DEVICE,
)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

print(
    "INFERENCE_BATCH_SIZE:",
    INFERENCE_BATCH_SIZE,
)

print(
    "STAGE_SERIES_LOCALLY:",
    STAGE_SERIES_LOCALLY,
)


# ============================================================
# 1. Preserve legacy implementation for equivalence audit
# ============================================================

if (
    "_run_frozen_sagittal_on_study_legacy"
    not in globals()
):
    _run_frozen_sagittal_on_study_legacy = (
        run_frozen_sagittal_on_study
    )

print(
    "Legacy inference preserved."
)


# ============================================================
# 2. Helpers
# ============================================================

def _resolve_sagittal_series_id(
    study_id_raw,
):

    rows = series_df[
        series_df["study_id"].astype(str)
        == str(study_id_raw)
    ]

    rows = rows[
        rows[
            "series_description"
        ]
        .astype(str)
        .str.strip()
        == PRIMARY_REFERENCE_SERIES
    ]

    if len(rows) == 0:
        return None

    return str(
        rows.iloc[0]["series_id"]
    )


def _stage_series_locally(
    study_id_raw,
    series_id_raw,
):

    source_dir = (
        TRAIN_IMAGES_ROOT
        / str(study_id_raw)
        / str(series_id_raw)
    )

    if not source_dir.is_dir():
        return source_dir, None

    if not STAGE_SERIES_LOCALLY:
        return source_dir, None

    study_stage_root = (
        LOCAL_STAGE_ROOT
        / opaque_id(study_id_raw)
    )

    staged_dir = (
        study_stage_root
        / opaque_id(series_id_raw)
    )

    if study_stage_root.exists():
        shutil.rmtree(
            study_stage_root,
            ignore_errors=True,
        )

    staged_dir.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        source_dir,
        staged_dir,
    )

    return (
        staged_dir,
        study_stage_root,
    )


def _finalize_67b2_consensus(
    predicted_components,
    anatomical_orientation_type,
    n_slices,
    series_id_raw,
):

    if not predicted_components:
        return {
            "status":
                "NO_DISC_COMPONENTS"
        }

    groups = consensus_union_find(
        [
            c["centroid_xyz"]
            for c in predicted_components
        ],
        distance_threshold_mm=15.0,
    )

    consensus_instances = []

    for indices in groups:

        members = [
            predicted_components[i]
            for i in indices
        ]

        centroids = np.stack(
            [
                m["centroid_xyz"]
                for m in members
            ]
        )

        mean_centroid = (
            centroids.mean(axis=0)
        )

        spread = (
            float(
                np.max(
                    np.linalg.norm(
                        centroids
                        - mean_centroid,
                        axis=1,
                    )
                )
            )
            if len(members) > 1
            else 0.0
        )

        mean_seg_conf = float(
            np.mean(
                [
                    m["mean_confidence"]
                    for m in members
                ]
            )
        )

        confidence_val = (
            compute_instance_confidence(
                len(members),
                spread,
                mean_seg_conf,
            )
        )

        consensus_instances.append({
            "centroid_xyz":
                mean_centroid,

            "supporting_slice_count":
                len(members),

            "instance_confidence":
                confidence_val,

            "centroid_spread_mm":
                spread,

            "mean_area_px":
                float(
                    np.mean(
                        [
                            m["area_px"]
                            for m in members
                        ]
                    )
                ),

            "border_touch":
                bool(
                    any(
                        m["border_touch"]
                        for m in members
                    )
                ),
        })

    if len(consensus_instances) < 2:
        return {
            "status":
                "INSUFFICIENT_CONSENSUS_INSTANCES",

            "predicted_instance_count":
                len(consensus_instances),
        }

    predicted_axis = (
        spine_axis_from_points(
            np.stack(
                [
                    c["centroid_xyz"]
                    for c
                    in consensus_instances
                ]
            )
        )
    )

    predicted_axis_reference_point = (
        consensus_instances[0][
            "centroid_xyz"
        ]
    )

    axis_positions = (
        project_points_onto_axis(
            [
                c["centroid_xyz"]
                for c
                in consensus_instances
            ],
            predicted_axis,
            predicted_axis_reference_point,
        )
    )

    for c, pos in zip(
        consensus_instances,
        axis_positions,
    ):
        c[
            "position_along_axis_mm"
        ] = pos

    consensus_instances.sort(
        key=lambda c:
            c[
                "position_along_axis_mm"
            ]
    )

    return {
        "status":
            "EXECUTED",

        "series_id_opaque":
            opaque_id(series_id_raw),

        "n_slices":
            int(n_slices),

        "predicted_instance_count":
            len(consensus_instances),

        "consensus_instances":
            consensus_instances,

        "predicted_axis":
            predicted_axis,

        "predicted_axis_reference_point":
            predicted_axis_reference_point,

        "anatomical_orientation_type":
            anatomical_orientation_type,
    }


# ============================================================
# 3. Optimized batched implementation
# ============================================================

def run_frozen_sagittal_on_study(
    study_id_raw,
):

    series_id_raw = (
        _resolve_sagittal_series_id(
            study_id_raw
        )
    )

    if series_id_raw is None:
        return {
            "status":
                "NO_SAGITTAL_T2_STIR"
        }

    staged_root = None

    try:

        working_series_dir, staged_root = (
            _stage_series_locally(
                study_id_raw,
                series_id_raw,
            )
        )

        dicom_paths = sorted(
            working_series_dir.glob(
                "*.dcm"
            )
        )

        if not dicom_paths:
            return {
                "status":
                    "NO_DICOM_FILES"
            }

        slices = (
            order_dicom_series_by_geometry(
                dicom_paths
            )
        )

        if len(slices) < 2:
            return {
                "status":
                    "INSUFFICIENT_SLICES"
            }

        anatomical_orientation_type = next(
            (
                s[
                    "anatomical_orientation_type"
                ]
                for s in slices
                if s.get(
                    "anatomical_orientation_type"
                )
            ),
            None,
        )

        target_size = tuple(
            sagittal_meta[
                "target_size"
            ]
        )

        predicted_components = []

        # ----------------------------------------------------
        # Batch sagittal slices.
        # ----------------------------------------------------

        for batch_start in range(
            0,
            len(slices),
            INFERENCE_BATCH_SIZE,
        ):

            batch_records = slices[
                batch_start:
                batch_start
                + INFERENCE_BATCH_SIZE
            ]

            prepared_batch = []
            native_shapes = []

            # -----------------------------------------------
            # CPU: DICOM decode + preprocessing
            # -----------------------------------------------

            for rec in batch_records:

                ds = pydicom.dcmread(
                    str(rec["path"])
                )

                native = (
                    ds.pixel_array
                    .astype(np.float32)
                )

                native_shapes.append(
                    native.shape
                )

                prepared = resize_image(
                    native,
                    target_size,
                )

                prepared_batch.append(
                    prepared
                )

            batch_np = np.stack(
                prepared_batch,
                axis=0,
            )

            batch_np = (
                batch_np[:, None, :, :]
            )

            tensor = (
                torch.from_numpy(
                    batch_np
                )
                .float()
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            # -----------------------------------------------
            # GPU: ONE forward for up to 32 slices.
            # -----------------------------------------------

            with torch.inference_mode():

                if DEVICE.type == "cuda":

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    ):

                        logits = (
                            sagittal_model(
                                tensor
                            )
                        )

                else:

                    logits = (
                        sagittal_model(
                            tensor
                        )
                    )

                probabilities = (
                    torch.softmax(
                        logits.float(),
                        dim=1,
                    )
                )

                predictions = (
                    torch.argmax(
                        probabilities,
                        dim=1,
                    )
                    .cpu()
                    .numpy()
                    .astype(np.uint8)
                )

                confidence_maps = (
                    torch.max(
                        probabilities,
                        dim=1,
                    )
                    .values
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

            # Release GPU tensor ASAP.
            del tensor
            del logits
            del probabilities

            # -----------------------------------------------
            # CPU post-processing remains slice-by-slice.
            # -----------------------------------------------

            for local_i, rec in enumerate(
                batch_records
            ):

                slice_index = (
                    batch_start
                    + local_i
                )

                native_shape = (
                    native_shapes[
                        local_i
                    ]
                )

                prediction = (
                    predictions[
                        local_i
                    ]
                )

                confidence_map = (
                    confidence_maps[
                        local_i
                    ]
                )

                disc_mask = (
                    prediction == 3
                )

                for component in (
                    connected_instances(
                        disc_mask
                    )
                ):

                    geom = (
                        component_geometry(
                            component
                        )
                    )

                    row_native = (
                        geom[
                            "centroid_row"
                        ]
                        * native_shape[0]
                        / target_size[0]
                    )

                    col_native = (
                        geom[
                            "centroid_col"
                        ]
                        * native_shape[1]
                        / target_size[1]
                    )

                    if not (
                        0
                        <= row_native
                        < rec["rows"]
                        and
                        0
                        <= col_native
                        < rec["columns"]
                        and
                        check_orientation_valid(
                            rec["iop"]
                        )
                    ):
                        continue

                    xyz = (
                        dicom_pixel_to_patient_xyz(
                            col_native,
                            row_native,
                            rec[
                                "pixel_spacing"
                            ],
                            rec["ipp"],
                            rec["iop"],
                        )
                    )

                    if not np.all(
                        np.isfinite(
                            xyz
                        )
                    ):
                        continue

                    predicted_components.append({
                        "slice_index":
                            slice_index,

                        "centroid_xyz":
                            xyz,

                        "area_px":
                            geom[
                                "area_px"
                            ],

                        "border_touch":
                            geom[
                                "border_touch"
                            ],

                        # EXACT same semantics
                        # as previous 67B2.
                        "mean_confidence":
                            float(
                                confidence_map.mean()
                            ),
                    })

        return (
            _finalize_67b2_consensus(
                predicted_components,
                anatomical_orientation_type,
                len(slices),
                series_id_raw,
            )
        )

    finally:

        if staged_root is not None:
            shutil.rmtree(
                staged_root,
                ignore_errors=True,
            )


print(
    "Optimized batched inference defined."
)


# ============================================================
# 4. HARD SMOKE PARITY
# ============================================================

EXPECTED_OPTIMIZED_SMOKE = {
    "4f06df2fd53b": 8,
    "d41a396f20c6": 9,
    "ef2ff5b618cf": 7,
}

_smoke_raw_map = {
    opaque_id(sid): sid
    for sid in eligible_validation_raw
}

print()
print(
    "=== L4 OPTIMIZED SMOKE CHECK ==="
)

for opaque, expected in (
    EXPECTED_OPTIMIZED_SMOKE.items()
):

    assert opaque in _smoke_raw_map

    raw_sid = _smoke_raw_map[
        opaque
    ]

    t_case = time.time()

    optimized = (
        run_frozen_sagittal_on_study(
            raw_sid
        )
    )

    observed = (
        optimized.get(
            "predicted_instance_count"
        )
        if optimized.get("status")
        == "EXECUTED"
        else None
    )

    elapsed_case = (
        time.time()
        - t_case
    )

    print(
        opaque,
        "| expected=",
        expected,
        "| observed=",
        observed,
        "| time=",
        round(
            elapsed_case,
            2,
        ),
        "sec",
    )

    assert observed == expected, (
        "HARD STOP: optimized batch "
        f"inference parity failed for "
        f"{opaque}: "
        f"expected={expected}, "
        f"observed={observed}"
    )


print()
print(
    "L4 OPTIMIZED SMOKE PARITY: PASS"
)

print(
    "Existing feature cache can be reused."
)

print(
    "Now rerun ONLY the feature extraction cell."
)

DEVICE: cuda
GPU: NVIDIA L4
INFERENCE_BATCH_SIZE: 32
STAGE_SERIES_LOCALLY: True
Legacy inference preserved.
Optimized batched inference defined.

=== L4 OPTIMIZED SMOKE CHECK ===
4f06df2fd53b | expected= 8 | observed= 8 | time= 0.42 sec
d41a396f20c6 | expected= 9 | observed= 9 | time= 0.5 sec
ef2ff5b618cf | expected= 7 | observed= 7 | time= 0.43 sec

L4 OPTIMIZED SMOKE PARITY: PASS
Existing feature cache can be reused.
Now rerun ONLY the feature extraction cell.


In [16]:
# ============================================================
# 67B2 — ULTRA-FAST L4 FEATURE EXTRACTION
#
# REPLACES CELL #10 COMPLETELY.
#
# Reuses existing Drive cache.
#
# Optimizations:
# - ONE DICOM read per slice
# - no per-study copytree to /content
# - parallel DICOM loading/preprocessing
# - cross-study GPU batching
# - no full confidence-map GPU->CPU transfer
# - GT physical XYZ reuses already-read DICOM metadata
#
# Scientific contract unchanged:
# - same frozen cf11 checkpoint
# - same preprocessing
# - same connected components
# - same 15 mm consensus
# - same 67B1 direction
# - same target
# - same features
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import gc
import time

# ------------------------------------------------------------
# PERFORMANCE CONFIG
# ------------------------------------------------------------

GPU_BATCH_SIZE = 64
STUDY_GROUP_SIZE = 16
IO_WORKERS = 8
CACHE_EVERY_GROUP = True

print("==============================================")
print("67B2 ULTRA-FAST EXTRACTION")
print("==============================================")
print("DEVICE:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("GPU_BATCH_SIZE:", GPU_BATCH_SIZE)
print("STUDY_GROUP_SIZE:", STUDY_GROUP_SIZE)
print("IO_WORKERS:", IO_WORKERS)

assert DEVICE.type == "cuda", (
    "This optimized path is intended for the L4 runtime."
)

torch.backends.cudnn.benchmark = True

# ------------------------------------------------------------
# Preserve previous implementation only for parity audit
# ------------------------------------------------------------

_previous_run_frozen = run_frozen_sagittal_on_study


# ============================================================
# 1 — Precompute study -> sagittal series
# ============================================================

_sag_rows_fast = series_df[
    series_df["series_description"]
    .astype(str)
    .str.strip()
    == PRIMARY_REFERENCE_SERIES
].copy()

_study_to_sag_series = {}

for _, row in _sag_rows_fast.iterrows():

    sid = str(row["study_id"])

    if sid not in _study_to_sag_series:
        _study_to_sag_series[sid] = str(
            row["series_id"]
        )

print(
    "Sagittal series map:",
    len(_study_to_sag_series),
)


# ============================================================
# 2 — SINGLE-PASS DICOM loader
# ============================================================

def _load_and_prepare_study_once(
    study_id_raw,
):

    sid = str(study_id_raw)

    series_id_raw = (
        _study_to_sag_series.get(sid)
    )

    if series_id_raw is None:
        return {
            "study_id_raw": sid,
            "status": "NO_SAGITTAL_T2_STIR",
        }

    series_dir = (
        TRAIN_IMAGES_ROOT
        / sid
        / series_id_raw
    )

    if not series_dir.is_dir():
        return {
            "study_id_raw": sid,
            "status": "NO_DICOM_FILES",
        }

    dicom_paths = list(
        series_dir.glob("*.dcm")
    )

    if not dicom_paths:
        return {
            "study_id_raw": sid,
            "status": "NO_DICOM_FILES",
        }

    target_size = tuple(
        sagittal_meta[
            "target_size"
        ]
    )

    slices = []

    try:

        for path in dicom_paths:

            # -----------------------------------------------
            # ONE read:
            # metadata + pixel data together
            # -----------------------------------------------

            ds = pydicom.dcmread(
                str(path)
            )

            required = [
                "Rows",
                "Columns",
                "PixelSpacing",
                "ImagePositionPatient",
                "ImageOrientationPatient",
                "InstanceNumber",
            ]

            if not all(
                hasattr(ds, tag)
                for tag in required
            ):
                continue

            iop = tuple(
                float(v)
                for v
                in ds.ImageOrientationPatient
            )

            if not check_orientation_valid(
                iop
            ):
                continue

            ipp = tuple(
                float(v)
                for v
                in ds.ImagePositionPatient
            )

            spacing = tuple(
                float(v)
                for v
                in ds.PixelSpacing
            )

            row_cosines = np.asarray(
                iop[:3],
                dtype=np.float64,
            )

            col_cosines = np.asarray(
                iop[3:],
                dtype=np.float64,
            )

            normal = np.cross(
                row_cosines,
                col_cosines,
            )

            position_mm = float(
                np.dot(
                    np.asarray(
                        ipp,
                        dtype=np.float64,
                    ),
                    normal,
                )
            )

            native = (
                ds.pixel_array
                .astype(np.float32)
            )

            prepared = resize_image(
                native,
                target_size,
            )

            slices.append({
                "instance_number":
                    int(ds.InstanceNumber),

                "position_mm":
                    position_mm,

                "ipp":
                    ipp,

                "iop":
                    iop,

                "pixel_spacing":
                    spacing,

                "rows":
                    int(ds.Rows),

                "columns":
                    int(ds.Columns),

                "native_shape":
                    native.shape,

                "prepared":
                    prepared,

                "anatomical_orientation_type":
                    getattr(
                        ds,
                        "AnatomicalOrientationType",
                        None,
                    ),
            })

    except Exception as exc:

        return {
            "study_id_raw": sid,
            "series_id_raw": series_id_raw,
            "status":
                f"DICOM_READ_ERROR:{type(exc).__name__}",
        }

    slices.sort(
        key=lambda x:
            x["position_mm"]
    )

    if len(slices) < 2:
        return {
            "study_id_raw": sid,
            "series_id_raw": series_id_raw,
            "status":
                "INSUFFICIENT_SLICES",
        }

    anatomical_orientation_type = next(
        (
            x[
                "anatomical_orientation_type"
            ]
            for x in slices
            if x.get(
                "anatomical_orientation_type"
            )
        ),
        None,
    )

    instance_map = {}

    duplicate_instances = set()

    for rec in slices:

        instance_number = (
            rec["instance_number"]
        )

        if instance_number in instance_map:
            duplicate_instances.add(
                instance_number
            )
        else:
            instance_map[
                instance_number
            ] = rec

    return {
        "study_id_raw":
            sid,

        "series_id_raw":
            series_id_raw,

        "status":
            "LOADED",

        "slices":
            slices,

        "instance_map":
            instance_map,

        "duplicate_instances":
            duplicate_instances,

        "anatomical_orientation_type":
            anatomical_orientation_type,
    }


# ============================================================
# 3 — Consensus finalizer
# ============================================================

def _finalize_ultrafast_consensus(
    loaded,
    predicted_components,
):

    if not predicted_components:

        return {
            "status":
                "NO_DISC_COMPONENTS"
        }

    groups = consensus_union_find(
        [
            x["centroid_xyz"]
            for x
            in predicted_components
        ],
        distance_threshold_mm=15.0,
    )

    consensus_instances = []

    for indices in groups:

        members = [
            predicted_components[i]
            for i in indices
        ]

        centroids = np.stack(
            [
                m["centroid_xyz"]
                for m in members
            ]
        )

        mean_centroid = (
            centroids.mean(axis=0)
        )

        spread = (
            float(
                np.max(
                    np.linalg.norm(
                        centroids
                        - mean_centroid,
                        axis=1,
                    )
                )
            )
            if len(members) > 1
            else 0.0
        )

        mean_seg_conf = float(
            np.mean(
                [
                    m[
                        "mean_confidence"
                    ]
                    for m in members
                ]
            )
        )

        consensus_instances.append({

            "centroid_xyz":
                mean_centroid,

            "supporting_slice_count":
                len(members),

            "instance_confidence":
                compute_instance_confidence(
                    len(members),
                    spread,
                    mean_seg_conf,
                ),

            "centroid_spread_mm":
                spread,

            "mean_area_px":
                float(
                    np.mean(
                        [
                            m["area_px"]
                            for m in members
                        ]
                    )
                ),

            "border_touch":
                bool(
                    any(
                        m["border_touch"]
                        for m in members
                    )
                ),
        })

    if len(consensus_instances) < 2:

        return {
            "status":
                "INSUFFICIENT_CONSENSUS_INSTANCES",

            "predicted_instance_count":
                len(consensus_instances),
        }

    predicted_axis = (
        spine_axis_from_points(
            np.stack(
                [
                    x["centroid_xyz"]
                    for x
                    in consensus_instances
                ]
            )
        )
    )

    reference_point = (
        consensus_instances[0][
            "centroid_xyz"
        ]
    )

    raw_positions = (
        project_points_onto_axis(
            [
                x["centroid_xyz"]
                for x
                in consensus_instances
            ],
            predicted_axis,
            reference_point,
        )
    )

    for inst, pos in zip(
        consensus_instances,
        raw_positions,
    ):

        inst[
            "position_along_axis_mm"
        ] = pos

    consensus_instances.sort(
        key=lambda x:
            x[
                "position_along_axis_mm"
            ]
    )

    return {

        "status":
            "EXECUTED",

        "series_id_opaque":
            opaque_id(
                loaded[
                    "series_id_raw"
                ]
            ),

        "n_slices":
            len(
                loaded["slices"]
            ),

        "predicted_instance_count":
            len(
                consensus_instances
            ),

        "consensus_instances":
            consensus_instances,

        "predicted_axis":
            predicted_axis,

        "predicted_axis_reference_point":
            reference_point,

        "anatomical_orientation_type":
            loaded[
                "anatomical_orientation_type"
            ],
    }


# ============================================================
# 4 — CROSS-STUDY GPU batching
# ============================================================

def _infer_loaded_studies_ultrafast(
    loaded_studies,
):

    valid_loaded = [
        x
        for x in loaded_studies
        if x.get("status")
        == "LOADED"
    ]

    results = {}

    # Preserve loader failures.
    for loaded in loaded_studies:

        sid = loaded[
            "study_id_raw"
        ]

        if loaded.get("status") != "LOADED":

            results[sid] = {
                "status":
                    loaded.get(
                        "status"
                    )
            }

    if not valid_loaded:
        return results

    # -----------------------------------------------
    # Flatten slices ACROSS STUDIES.
    # -----------------------------------------------

    flat = []

    for loaded in valid_loaded:

        sid = loaded[
            "study_id_raw"
        ]

        for slice_index, rec in enumerate(
            loaded["slices"]
        ):

            flat.append({
                "study_id_raw":
                    sid,

                "slice_index":
                    slice_index,

                "rec":
                    rec,
            })

    components_by_study = {
        loaded[
            "study_id_raw"
        ]: []
        for loaded
        in valid_loaded
    }

    # -----------------------------------------------
    # Large GPU batches
    # -----------------------------------------------

    for start in range(
        0,
        len(flat),
        GPU_BATCH_SIZE,
    ):

        items = flat[
            start:
            start + GPU_BATCH_SIZE
        ]

        batch_np = np.stack(
            [
                item[
                    "rec"
                ][
                    "prepared"
                ]
                for item in items
            ],
            axis=0,
        )[:, None, :, :]

        tensor = (
            torch.from_numpy(
                batch_np
            )
            .float()
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        with torch.inference_mode():

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                logits = (
                    sagittal_model(
                        tensor
                    )
                )

            # Same numerical semantics:
            # softmax in FP32.
            probabilities = (
                torch.softmax(
                    logits.float(),
                    dim=1,
                )
            )

            # We only need the winning class map.
            predictions = (
                torch.argmax(
                    probabilities,
                    dim=1,
                )
            )

            # IMPORTANT optimization:
            # previous code transferred the complete
            # confidence map to CPU only to compute mean().
            #
            # Compute mean on GPU and transfer ONE scalar.
            confidence_means = (
                torch.max(
                    probabilities,
                    dim=1,
                )
                .values
                .mean(
                    dim=(1, 2)
                )
            )

            predictions_cpu = (
                predictions
                .to(
                    dtype=torch.uint8
                )
                .cpu()
                .numpy()
            )

            confidence_means_cpu = (
                confidence_means
                .cpu()
                .numpy()
                .astype(np.float32)
            )

        del tensor
        del logits
        del probabilities
        del predictions
        del confidence_means

        # -----------------------------------------------
        # Per-slice connected components / geometry
        # -----------------------------------------------

        for j, item in enumerate(
            items
        ):

            sid = item[
                "study_id_raw"
            ]

            rec = item["rec"]

            prediction = (
                predictions_cpu[j]
            )

            mean_confidence = float(
                confidence_means_cpu[j]
            )

            disc_mask = (
                prediction == 3
            )

            for component in (
                connected_instances(
                    disc_mask
                )
            ):

                geom = (
                    component_geometry(
                        component
                    )
                )

                native_shape = (
                    rec[
                        "native_shape"
                    ]
                )

                target_size = tuple(
                    sagittal_meta[
                        "target_size"
                    ]
                )

                row_native = (
                    geom[
                        "centroid_row"
                    ]
                    * native_shape[0]
                    / target_size[0]
                )

                col_native = (
                    geom[
                        "centroid_col"
                    ]
                    * native_shape[1]
                    / target_size[1]
                )

                if not (
                    0
                    <= row_native
                    < rec["rows"]
                    and
                    0
                    <= col_native
                    < rec["columns"]
                ):
                    continue

                xyz = (
                    dicom_pixel_to_patient_xyz(
                        col_native,
                        row_native,
                        rec[
                            "pixel_spacing"
                        ],
                        rec["ipp"],
                        rec["iop"],
                    )
                )

                if not np.all(
                    np.isfinite(
                        xyz
                    )
                ):
                    continue

                components_by_study[
                    sid
                ].append({

                    "slice_index":
                        item[
                            "slice_index"
                        ],

                    "centroid_xyz":
                        xyz,

                    "area_px":
                        geom[
                            "area_px"
                        ],

                    "border_touch":
                        geom[
                            "border_touch"
                        ],

                    "mean_confidence":
                        mean_confidence,
                })

    # -----------------------------------------------
    # Finalize each study independently.
    # -----------------------------------------------

    loaded_by_sid = {
        x[
            "study_id_raw"
        ]: x
        for x
        in valid_loaded
    }

    for sid, loaded in (
        loaded_by_sid.items()
    ):

        results[sid] = (
            _finalize_ultrafast_consensus(
                loaded,
                components_by_study[
                    sid
                ],
            )
        )

    return results


# ============================================================
# 5 — GT from already-loaded metadata
# ============================================================

def _resolve_gt_from_loaded_study(
    study_id_raw,
    loaded,
):

    rows = primary_reference_df[
        primary_reference_df[
            "study_id"
        ]
        .astype(str)
        == str(study_id_raw)
    ]

    result = {}

    for _, row in rows.iterrows():

        level = normalize_level(
            row[
                "level_raw"
            ]
        )

        if level not in CANONICAL_LEVELS:
            continue

        instance_number = int(
            row[
                "instance_number"
            ]
        )

        # Rare ambiguous InstanceNumber:
        # fail over to existing audited resolver.
        if (
            instance_number
            in loaded[
                "duplicate_instances"
            ]
        ):

            return (
                resolve_gt_level_xyz_for_study(
                    study_id_raw
                )
            )

        rec = (
            loaded[
                "instance_map"
            ]
            .get(
                instance_number
            )
        )

        if rec is None:
            continue

        xyz = (
            dicom_pixel_to_patient_xyz(
                float(row["x"]),
                float(row["y"]),
                rec[
                    "pixel_spacing"
                ],
                rec["ipp"],
                rec["iop"],
            )
        )

        if np.all(
            np.isfinite(
                xyz
            )
        ):

            result[level] = xyz

    return result


# ============================================================
# 6 — SMOKE HARD PARITY BEFORE MIXING CACHE
# ============================================================

EXPECTED_SMOKE = {
    "4f06df2fd53b": 8,
    "d41a396f20c6": 9,
    "ef2ff5b618cf": 7,
}

_smoke_raw = {
    opaque_id(sid): sid
    for sid
    in eligible_validation_raw
}

_smoke_ids = [
    _smoke_raw[x]
    for x
    in EXPECTED_SMOKE
]

print()
print(
    "Running ultra-fast smoke parity..."
)

with ThreadPoolExecutor(
    max_workers=min(
        IO_WORKERS,
        3,
    )
) as executor:

    smoke_loaded = list(
        executor.map(
            _load_and_prepare_study_once,
            _smoke_ids,
        )
    )

smoke_ultrafast = (
    _infer_loaded_studies_ultrafast(
        smoke_loaded
    )
)

for opaque, expected in (
    EXPECTED_SMOKE.items()
):

    sid = str(
        _smoke_raw[
            opaque
        ]
    )

    result = (
        smoke_ultrafast[
            sid
        ]
    )

    observed = (
        result.get(
            "predicted_instance_count"
        )
        if result.get(
            "status"
        )
        == "EXECUTED"
        else None
    )

    print(
        opaque,
        "expected=",
        expected,
        "observed=",
        observed,
    )

    assert (
        observed == expected
    ), (
        "HARD STOP: ultra-fast path "
        f"failed smoke parity for {opaque}"
    )

print(
    "ULTRA-FAST SMOKE PARITY: PASS"
)


# ============================================================
# 7 — LOAD EXISTING CACHE
# ============================================================

FEATURE_CACHE_CSV = (
    RESULTS_DIR
    / "window_features_outer_train.csv"
)

STUDY_CACHE_CSV = (
    RESULTS_DIR
    / "window_study_summary_outer_train.csv"
)

if FEATURE_CACHE_CSV.is_file():

    existing_feature_df = (
        pd.read_csv(
            FEATURE_CACHE_CSV
        )
    )

else:

    existing_feature_df = (
        pd.DataFrame()
    )

if STUDY_CACHE_CSV.is_file():

    existing_study_df = (
        pd.read_csv(
            STUDY_CACHE_CSV
        )
    )

else:

    existing_study_df = (
        pd.DataFrame()
    )

processed_opaque = set(
    existing_study_df[
        "study_id_opaque"
    ]
    .astype(str)
    .tolist()
    if (
        len(existing_study_df)
        and
        "study_id_opaque"
        in existing_study_df.columns
    )
    else []
)

print()
print(
    "Previously processed studies:",
    len(processed_opaque),
)

print(
    "Eligible TRAIN studies:",
    len(eligible_train_raw),
)

remaining_raw = [
    sid
    for sid in eligible_train_raw
    if opaque_id(sid)
    not in processed_opaque
]

print(
    "Remaining:",
    len(remaining_raw),
)


# ============================================================
# 8 — CACHE FLUSH
# ============================================================

feature_rows = []
study_rows = []


def _flush_ultrafast_cache():

    global existing_feature_df
    global existing_study_df
    global feature_rows
    global study_rows

    if feature_rows:

        new_features = (
            pd.DataFrame(
                feature_rows
            )
        )

        existing_feature_df = pd.concat(
            [
                existing_feature_df,
                new_features,
            ],
            ignore_index=True,
        )

        existing_feature_df = (
            existing_feature_df
            .drop_duplicates(
                subset=[
                    "study_id_opaque",
                    "candidate_window_index",
                ],
                keep="last",
            )
        )

        existing_feature_df.to_csv(
            FEATURE_CACHE_CSV,
            index=False,
        )

        feature_rows = []

    if study_rows:

        new_studies = (
            pd.DataFrame(
                study_rows
            )
        )

        existing_study_df = pd.concat(
            [
                existing_study_df,
                new_studies,
            ],
            ignore_index=True,
        )

        existing_study_df = (
            existing_study_df
            .drop_duplicates(
                subset=[
                    "study_id_opaque"
                ],
                keep="last",
            )
        )

        existing_study_df.to_csv(
            STUDY_CACHE_CSV,
            index=False,
        )

        study_rows = []


# ============================================================
# 9 — ULTRA-FAST EXTRACTION LOOP
# ============================================================

t0 = time.time()
newly_processed = 0

for group_start in range(
    0,
    len(remaining_raw),
    STUDY_GROUP_SIZE,
):

    group_ids = remaining_raw[
        group_start:
        group_start
        + STUDY_GROUP_SIZE
    ]

    # --------------------------------------------------------
    # Parallel CPU / Drive DICOM load + preprocessing
    # --------------------------------------------------------

    with ThreadPoolExecutor(
        max_workers=IO_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                _load_and_prepare_study_once,
                sid,
            ): sid
            for sid in group_ids
        }

        loaded_group = []

        for future in as_completed(
            futures
        ):

            sid = futures[
                future
            ]

            try:

                loaded_group.append(
                    future.result()
                )

            except Exception as exc:

                loaded_group.append({
                    "study_id_raw":
                        str(sid),

                    "status":
                        f"LOAD_ERROR:{type(exc).__name__}",
                })

    # Restore deterministic study order.
    loaded_map = {
        x[
            "study_id_raw"
        ]: x
        for x
        in loaded_group
    }

    loaded_group = [
        loaded_map[
            str(sid)
        ]
        for sid
        in group_ids
    ]

    # --------------------------------------------------------
    # Cross-study batched GPU inference
    # --------------------------------------------------------

    group_results = (
        _infer_loaded_studies_ultrafast(
            loaded_group
        )
    )

    loaded_by_sid = {
        x[
            "study_id_raw"
        ]: x
        for x
        in loaded_group
    }

    # --------------------------------------------------------
    # Target + features
    # --------------------------------------------------------

    for sid_raw in group_ids:

        sid = str(
            sid_raw
        )

        opaque = opaque_id(
            sid
        )

        loaded = (
            loaded_by_sid[
                sid
            ]
        )

        run = (
            group_results[
                sid
            ]
        )

        summary = {

            "study_id_opaque":
                opaque,

            "runtime_status":
                run.get(
                    "status"
                ),

            "predicted_instance_count":
                run.get(
                    "predicted_instance_count"
                ),

            "direction_status":
                None,

            "target_qc_status":
                None,

            "target_window_index":
                None,

            "n_candidate_windows":
                0,

            "eligible_for_scorer_training":
                False,
        }

        if (
            run.get("status")
            != "EXECUTED"
        ):

            study_rows.append(
                summary
            )

            processed_opaque.add(
                opaque
            )

            newly_processed += 1

            continue

        direction, ordered_instances = (
            orient_and_sort_instances(
                run
            )
        )

        summary[
            "direction_status"
        ] = direction.get(
            "status"
        )

        if (
            direction.get(
                "status"
            )
            != "RESOLVED"
        ):

            summary[
                "target_qc_status"
            ] = (
                "DIRECTION_UNRESOLVED"
            )

            study_rows.append(
                summary
            )

            processed_opaque.add(
                opaque
            )

            newly_processed += 1

            continue

        windows = (
            enumerate_candidate_windows(
                len(
                    ordered_instances
                )
            )
        )

        summary[
            "n_candidate_windows"
        ] = len(
            windows
        )

        if not windows:

            summary[
                "target_qc_status"
            ] = "N_LT_5"

            study_rows.append(
                summary
            )

            processed_opaque.add(
                opaque
            )

            newly_processed += 1

            continue

        # ----------------------------------------------------
        # Reuse already-loaded DICOM geometry for GT.
        # ZERO extra DICOM reads in normal case.
        # ----------------------------------------------------

        gt = (
            _resolve_gt_from_loaded_study(
                sid,
                loaded,
            )
        )

        target_info = (
            gt_target_qc_and_best_window(
                windows,
                ordered_instances,
                direction[
                    "oriented_axis"
                ],
                run[
                    "predicted_axis_reference_point"
                ],
                gt,
            )
        )

        summary[
            "target_qc_status"
        ] = target_info[
            "status"
        ]

        summary[
            "target_window_index"
        ] = target_info.get(
            "target_window_index"
        )

        summary[
            "eligible_for_scorer_training"
        ] = bool(
            target_info[
                "status"
            ]
            == "TARGET_OK"
            and
            len(windows) >= 2
        )

        if (
            target_info[
                "status"
            ]
            == "TARGET_OK"
        ):

            for w in windows:

                features = (
                    build_window_feature_row(
                        w,
                        ordered_instances,
                        direction[
                            "oriented_axis"
                        ],
                        run[
                            "predicted_axis_reference_point"
                        ],
                    )
                )

                feature_rows.append({

                    "study_id_opaque":
                        opaque,

                    "candidate_window_index":
                        int(
                            w[
                                "window_index"
                            ]
                        ),

                    "target_window_index":
                        int(
                            target_info[
                                "target_window_index"
                            ]
                        ),

                    "is_target":
                        int(
                            w[
                                "window_index"
                            ]
                            ==
                            target_info[
                                "target_window_index"
                            ]
                        ),

                    **features,
                })

        study_rows.append(
            summary
        )

        processed_opaque.add(
            opaque
        )

        newly_processed += 1

    # --------------------------------------------------------
    # Flush EVERY group
    # Maximum loss if Colab dies: <= 16 studies.
    # --------------------------------------------------------

    _flush_ultrafast_cache()

    # Free group memory aggressively.
    del loaded_group
    del loaded_map
    del loaded_by_sid
    del group_results

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    elapsed = (
        time.time()
        - t0
    )

    total_done = len(
        processed_opaque
    )

    rate = (
        newly_processed
        /
        max(
            elapsed / 60.0,
            1e-9,
        )
    )

    remaining_count = (
        len(
            eligible_train_raw
        )
        -
        total_done
    )

    eta = (
        remaining_count
        /
        max(
            rate,
            1e-9,
        )
    )

    print(
        f"Processed "
        f"{total_done}/{len(eligible_train_raw)} "
        f"| new={newly_processed} "
        f"| elapsed={elapsed/60:.1f} min "
        f"| rate={rate:.2f} studies/min "
        f"| ETA={eta:.1f} min"
    )


# ============================================================
# 10 — FINAL CACHE LOAD
# ============================================================

feature_df = pd.read_csv(
    FEATURE_CACHE_CSV
)

study_df = pd.read_csv(
    STUDY_CACHE_CSV
)

print()
print(
    "=============================================="
)

print(
    "ULTRA-FAST FEATURE EXTRACTION COMPLETE"
)

print(
    "=============================================="
)

print(
    "Feature rows:",
    len(feature_df),
)

print(
    "Study rows:",
    len(study_df),
)

print()

print(
    "runtime_status:"
)

print(
    study_df[
        "runtime_status"
    ]
    .value_counts(
        dropna=False
    )
)

print()

print(
    "target_qc_status:"
)

print(
    study_df[
        "target_qc_status"
    ]
    .value_counts(
        dropna=False
    )
)

print()

print(
    "Eligible for scorer training:",
    int(
        study_df[
            "eligible_for_scorer_training"
        ]
        .astype(bool)
        .sum()
    ),
)

assert len(study_df) == len(
    eligible_train_raw
)

assert "study_id" not in feature_df.columns
assert "study_id" not in study_df.columns

print()
print(
    "DONE — continue directly with cell #11."
)

67B2 ULTRA-FAST EXTRACTION
DEVICE: cuda
GPU: NVIDIA L4
GPU_BATCH_SIZE: 64
STUDY_GROUP_SIZE: 16
IO_WORKERS: 8
Sagittal series map: 1974

Running ultra-fast smoke parity...
4f06df2fd53b expected= 8 observed= 8
d41a396f20c6 expected= 9 observed= 9
ef2ff5b618cf expected= 7 observed= 7
ULTRA-FAST SMOKE PARITY: PASS

Previously processed studies: 600
Eligible TRAIN studies: 1322
Remaining: 722
Processed 616/1322 | new=16 | elapsed=0.6 min | rate=28.76 studies/min | ETA=24.5 min
Processed 632/1322 | new=32 | elapsed=1.4 min | rate=23.64 studies/min | ETA=29.2 min
Processed 648/1322 | new=48 | elapsed=2.2 min | rate=21.37 studies/min | ETA=31.5 min
Processed 664/1322 | new=64 | elapsed=3.0 min | rate=21.68 studies/min | ETA=30.4 min
Processed 680/1322 | new=80 | elapsed=3.7 min | rate=21.34 studies/min | ETA=30.1 min
Processed 696/1322 | new=96 | elapsed=4.5 min | rate=21.43 studies/min | ETA=29.2 min
Processed 712/1322 | new=112 | elapsed=5.2 min | rate=21.41 studies/min | ETA=28.5 min
Proces

In [17]:
# ============================================================
# 11 — Build deterministic scorer TRAIN / DEV split
# ============================================================

eligible_scorer_studies = sorted(
    study_df.loc[
        study_df["eligible_for_scorer_training"].astype(bool),
        "study_id_opaque",
    ].astype(str).unique()
)

assert len(eligible_scorer_studies) > 0, (
    "No eligible scorer studies. Feature extraction likely incomplete."
)

rng = np.random.default_rng(SEED)
shuffled = list(eligible_scorer_studies)
rng.shuffle(shuffled)

n_dev = max(1, int(round(0.20 * len(shuffled))))
dev_studies = set(shuffled[:n_dev])
train_studies = set(shuffled[n_dev:])

assert train_studies.isdisjoint(dev_studies)

scorer_df = feature_df[
    feature_df["study_id_opaque"].astype(str).isin(
        train_studies | dev_studies
    )
].copy()

scorer_df["scorer_split"] = np.where(
    scorer_df["study_id_opaque"].astype(str).isin(dev_studies),
    "dev",
    "train",
)

NON_FEATURE_COLUMNS = {
    "study_id_opaque",
    "candidate_window_index",
    "target_window_index",
    "is_target",
    "scorer_split",
}

FEATURE_NAMES = [
    c for c in scorer_df.columns
    if c not in NON_FEATURE_COLUMNS
]

# Strict numeric feature contract.
for c in FEATURE_NAMES:
    scorer_df[c] = pd.to_numeric(
        scorer_df[c],
        errors="raise",
    )

assert np.isfinite(
    scorer_df[FEATURE_NAMES].to_numpy(dtype=np.float64)
).all()

print("Eligible scorer studies:", len(eligible_scorer_studies))
print("scorer train studies:", len(train_studies))
print("scorer dev studies:", len(dev_studies))
print("features:", len(FEATURE_NAMES))
print(FEATURE_NAMES)

split_manifest = {
    "seed": SEED,
    "outer_split": "TRAIN_ONLY",
    "scorer_train_studies": len(train_studies),
    "scorer_dev_studies": len(dev_studies),
    "feature_names": FEATURE_NAMES,
    "validation_external_accessed": False,
    "internal_test_accessed": False,
    "official_test_accessed": False,
}

(METRICS_DIR / "scorer_split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2),
    encoding="utf-8",
)

Eligible scorer studies: 1295
scorer train studies: 1036
scorer dev studies: 259
features: 49
['n_instances', 'n_candidate_windows', 'window_index_norm', 'rank_start_norm', 'rank_end_norm', 'left_margin_count', 'right_margin_count', 'min_margin_count', 'margin_balance_abs', 'window_span_mm', 'global_span_mm', 'window_center_offset_from_global_center_mm', 'gap_mean_mm', 'gap_std_mm', 'gap_cv', 'gap_min_mm', 'gap_max_mm', 'gap_range_mm', 'gap_1_mm', 'gap_2_mm', 'gap_3_mm', 'gap_4_mm', 'left_outside_gap_available', 'right_outside_gap_available', 'left_outside_gap_mm', 'right_outside_gap_mm', 'left_boundary_gap_ratio', 'right_boundary_gap_ratio', 'confidence_mean', 'confidence_std', 'confidence_min', 'confidence_max', 'support_mean', 'support_std', 'support_min', 'support_max', 'spread_mean_mm', 'spread_std_mm', 'spread_max_mm', 'area_mean_px', 'area_std_px', 'area_min_px', 'area_max_px', 'log1p_area_mean', 'border_touch_count', 'border_touch_fraction', 'axis_residual_mean_mm', 'axis_resid

1384

In [18]:
# ============================================================
# 12 — Standardization (fit on scorer TRAIN only)
# ============================================================

train_candidate_df = scorer_df[
    scorer_df["scorer_split"] == "train"
].copy()

dev_candidate_df = scorer_df[
    scorer_df["scorer_split"] == "dev"
].copy()

feature_mean = (
    train_candidate_df[FEATURE_NAMES]
    .mean(axis=0)
    .to_numpy(dtype=np.float32)
)

feature_std = (
    train_candidate_df[FEATURE_NAMES]
    .std(axis=0, ddof=0)
    .to_numpy(dtype=np.float32)
)

feature_std = np.where(
    feature_std < 1e-8,
    1.0,
    feature_std,
).astype(np.float32)

print("Scaler fit on scorer TRAIN candidate rows only.")
print("mean shape:", feature_mean.shape)
print("std shape:", feature_std.shape)

Scaler fit on scorer TRAIN candidate rows only.
mean shape: (49,)
std shape: (49,)


In [19]:
# ============================================================
# 13 — Study-level ranking dataset
# ============================================================

class WindowRankingDataset(Dataset):
    def __init__(
        self,
        df,
        study_ids,
        feature_names,
        mean,
        std,
    ):
        self.df = df.copy()
        self.study_ids = sorted(str(x) for x in study_ids)
        self.feature_names = list(feature_names)
        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)

        self.groups = {}

        for sid in self.study_ids:
            g = self.df[
                self.df["study_id_opaque"].astype(str) == sid
            ].sort_values("candidate_window_index")

            if len(g) < 2:
                continue

            target_window = int(
                g["target_window_index"].iloc[0]
            )

            candidate_indices = (
                g["candidate_window_index"]
                .astype(int)
                .tolist()
            )

            assert target_window in candidate_indices

            target_position = candidate_indices.index(
                target_window
            )

            x = g[self.feature_names].to_numpy(
                dtype=np.float32
            )

            x = (x - self.mean) / self.std

            self.groups[sid] = {
                "x": x,
                "target_position": target_position,
                "candidate_indices": candidate_indices,
            }

        self.study_ids = [
            sid for sid in self.study_ids
            if sid in self.groups
        ]

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        sid = self.study_ids[idx]
        item = self.groups[sid]
        return {
            "study_id_opaque": sid,
            "x": torch.from_numpy(item["x"]),
            "target_position":
                int(item["target_position"]),
            "candidate_indices":
                item["candidate_indices"],
        }

def collate_window_ranking(batch):
    b = len(batch)
    max_k = max(item["x"].shape[0] for item in batch)
    d = batch[0]["x"].shape[1]

    x = torch.zeros(
        b, max_k, d,
        dtype=torch.float32,
    )

    mask = torch.zeros(
        b, max_k,
        dtype=torch.bool,
    )

    targets = torch.zeros(
        b,
        dtype=torch.long,
    )

    study_ids = []
    candidate_indices = []

    for i, item in enumerate(batch):
        k = item["x"].shape[0]
        x[i, :k] = item["x"]
        mask[i, :k] = True
        targets[i] = item["target_position"]
        study_ids.append(item["study_id_opaque"])
        candidate_indices.append(item["candidate_indices"])

    return {
        "x": x,
        "mask": mask,
        "targets": targets,
        "study_ids": study_ids,
        "candidate_indices": candidate_indices,
    }

train_dataset = WindowRankingDataset(
    scorer_df,
    train_studies,
    FEATURE_NAMES,
    feature_mean,
    feature_std,
)

dev_dataset = WindowRankingDataset(
    scorer_df,
    dev_studies,
    FEATURE_NAMES,
    feature_mean,
    feature_std,
)

print("train dataset studies:", len(train_dataset))
print("dev dataset studies:", len(dev_dataset))

assert len(train_dataset) == len(train_studies)
assert len(dev_dataset) == len(dev_studies)

train dataset studies: 1036
dev dataset studies: 259


In [20]:
# ============================================================
# 14 — Window scorer MLP + evaluation
# ============================================================

class AbsoluteLumbarWindowScorer(nn.Module):
    def __init__(self, input_dim, hidden1=32, hidden2=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        # x: [B, K, D]
        return self.net(x).squeeze(-1)

def masked_logits(model, x, mask):
    logits = model(x)
    return logits.masked_fill(~mask, -1e9)

@torch.no_grad()
def evaluate_ranking(model, loader):
    model.eval()

    rows = []
    total_loss = 0.0
    total_studies = 0
    criterion = nn.CrossEntropyLoss(reduction="sum")

    for batch in loader:
        x = batch["x"].to(DEVICE, non_blocking=True)
        mask = batch["mask"].to(DEVICE, non_blocking=True)
        targets = batch["targets"].to(DEVICE, non_blocking=True)

        logits = masked_logits(model, x, mask)

        loss = criterion(logits, targets)
        total_loss += float(loss.item())
        total_studies += int(x.shape[0])

        probs = torch.softmax(logits, dim=1)

        for i in range(x.shape[0]):
            k = int(mask[i].sum().item())
            valid_probs = probs[i, :k]
            target = int(targets[i].item())

            order = torch.argsort(
                valid_probs,
                descending=True,
            )

            rank = (
                int(
                    (order == target)
                    .nonzero(as_tuple=False)[0]
                    .item()
                )
                + 1
            )

            top1_pos = int(order[0].item())
            top1_prob = float(valid_probs[top1_pos].item())

            if k >= 2:
                top2_pos = int(order[1].item())
                top2_prob = float(valid_probs[top2_pos].item())
                margin = top1_prob - top2_prob
            else:
                top2_prob = 0.0
                margin = 1.0

            candidate_indices = batch["candidate_indices"][i]

            rows.append({
                "study_id_opaque":
                    batch["study_ids"][i],
                "n_candidates": k,
                "target_window_index":
                    int(candidate_indices[target]),
                "predicted_window_index":
                    int(candidate_indices[top1_pos]),
                "rank_of_target": rank,
                "top1_correct":
                    int(top1_pos == target),
                "top2_correct":
                    int(rank <= 2),
                "top1_probability": top1_prob,
                "top2_probability": top2_prob,
                "top1_top2_margin": float(margin),
            })

    result_df = pd.DataFrame(rows)

    metrics = {
        "loss":
            total_loss / max(total_studies, 1),
        "top1_accuracy":
            float(result_df["top1_correct"].mean()),
        "top2_accuracy":
            float(result_df["top2_correct"].mean()),
        "mrr":
            float(
                np.mean(
                    1.0
                    / result_df["rank_of_target"].to_numpy(
                        dtype=np.float64
                    )
                )
            ),
        "n_studies":
            int(len(result_df)),
    }

    return metrics, result_df

print("Model + evaluation helpers: READY")

Model + evaluation helpers: READY


In [21]:
# ============================================================
# 15 — Train scorer
# ============================================================

BATCH_SIZE = 64
MAX_EPOCHS = 120
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 18

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_window_ranking,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_window_ranking,
)

scorer_model = AbsoluteLumbarWindowScorer(
    input_dim=len(FEATURE_NAMES),
    hidden1=32,
    hidden2=16,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    scorer_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

criterion = nn.CrossEntropyLoss()

best_state = None
best_epoch = None
best_dev_top1 = -1.0
best_dev_loss = float("inf")
epochs_without_improvement = 0
history = []

if TRAIN_SCORER:
    for epoch in range(1, MAX_EPOCHS + 1):
        scorer_model.train()

        train_loss_sum = 0.0
        train_studies_seen = 0

        for batch in train_loader:
            x = batch["x"].to(
                DEVICE,
                non_blocking=True,
            )
            mask = batch["mask"].to(
                DEVICE,
                non_blocking=True,
            )
            targets = batch["targets"].to(
                DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            logits = masked_logits(
                scorer_model,
                x,
                mask,
            )

            loss = criterion(
                logits,
                targets,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                scorer_model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            train_loss_sum += (
                float(loss.item())
                * int(x.shape[0])
            )
            train_studies_seen += int(x.shape[0])

        train_metrics, _ = evaluate_ranking(
            scorer_model,
            train_loader,
        )

        dev_metrics, _ = evaluate_ranking(
            scorer_model,
            dev_loader,
        )

        train_epoch_loss = (
            train_loss_sum
            / max(train_studies_seen, 1)
        )

        history.append({
            "epoch": epoch,
            "train_optimization_loss":
                train_epoch_loss,
            "train_top1":
                train_metrics["top1_accuracy"],
            "train_mrr":
                train_metrics["mrr"],
            "dev_loss":
                dev_metrics["loss"],
            "dev_top1":
                dev_metrics["top1_accuracy"],
            "dev_top2":
                dev_metrics["top2_accuracy"],
            "dev_mrr":
                dev_metrics["mrr"],
        })

        improved = (
            dev_metrics["top1_accuracy"]
            > best_dev_top1 + 1e-12
        ) or (
            abs(
                dev_metrics["top1_accuracy"]
                - best_dev_top1
            ) <= 1e-12
            and dev_metrics["loss"]
            < best_dev_loss
        )

        if improved:
            best_dev_top1 = (
                dev_metrics["top1_accuracy"]
            )
            best_dev_loss = dev_metrics["loss"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v
                in scorer_model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epoch == 1
            or epoch % 5 == 0
            or improved
        ):
            print(
                f"epoch={epoch:03d} "
                f"train_top1={train_metrics['top1_accuracy']:.4f} "
                f"dev_top1={dev_metrics['top1_accuracy']:.4f} "
                f"dev_top2={dev_metrics['top2_accuracy']:.4f} "
                f"dev_mrr={dev_metrics['mrr']:.4f} "
                f"dev_loss={dev_metrics['loss']:.4f}"
            )

        if epochs_without_improvement >= PATIENCE:
            print(
                "Early stopping at epoch",
                epoch,
                "| best epoch:",
                best_epoch,
            )
            break

    assert best_state is not None
    scorer_model.load_state_dict(
        best_state,
        strict=True,
    )
else:
    print("TRAIN_SCORER=False")

epoch=001 train_top1=0.8012 dev_top1=0.7954 dev_top2=0.9151 dev_mrr=0.8805 dev_loss=1.0345
epoch=003 train_top1=0.8031 dev_top1=0.8031 dev_top2=0.9382 dev_mrr=0.8891 dev_loss=0.5272
epoch=004 train_top1=0.8523 dev_top1=0.8687 dev_top2=0.9537 dev_mrr=0.9239 dev_loss=0.4187
epoch=005 train_top1=0.8755 dev_top1=0.8880 dev_top2=0.9614 dev_mrr=0.9357 dev_loss=0.3609
epoch=007 train_top1=0.8996 dev_top1=0.8880 dev_top2=0.9575 dev_mrr=0.9361 dev_loss=0.2922
epoch=008 train_top1=0.9054 dev_top1=0.8919 dev_top2=0.9614 dev_mrr=0.9385 dev_loss=0.2689
epoch=009 train_top1=0.9112 dev_top1=0.9073 dev_top2=0.9691 dev_mrr=0.9476 dev_loss=0.2497
epoch=010 train_top1=0.9228 dev_top1=0.9228 dev_top2=0.9653 dev_mrr=0.9546 dev_loss=0.2369
epoch=011 train_top1=0.9266 dev_top1=0.9266 dev_top2=0.9691 dev_mrr=0.9572 dev_loss=0.2279
epoch=012 train_top1=0.9276 dev_top1=0.9344 dev_top2=0.9730 dev_mrr=0.9617 dev_loss=0.2191
epoch=015 train_top1=0.9431 dev_top1=0.9266 dev_top2=0.9730 dev_mrr=0.9585 dev_loss=0.2042

In [22]:
# ============================================================
# 16 — Final TRAIN/DEV metrics + selective DEV curve
# ============================================================

final_train_metrics, final_train_predictions = evaluate_ranking(
    scorer_model,
    train_loader,
)

final_dev_metrics, final_dev_predictions = evaluate_ranking(
    scorer_model,
    dev_loader,
)

print("FINAL TRAIN:")
print(json.dumps(final_train_metrics, indent=2))
print()
print("FINAL DEV:")
print(json.dumps(final_dev_metrics, indent=2))

# Per-number-of-candidates diagnostic.
per_candidate_count = (
    final_dev_predictions
    .groupby("n_candidates")
    .agg(
        studies=("study_id_opaque", "count"),
        top1_accuracy=("top1_correct", "mean"),
        top2_accuracy=("top2_correct", "mean"),
        mean_margin=("top1_top2_margin", "mean"),
    )
    .reset_index()
)

print()
print("DEV by candidate count:")
print(per_candidate_count.to_string(index=False))

# Selective curve: descriptive only.
# No threshold is frozen here.
margins = final_dev_predictions[
    "top1_top2_margin"
].to_numpy(dtype=np.float64)

quantiles = np.unique(
    np.quantile(
        margins,
        np.linspace(0.0, 1.0, 21),
    )
)

selective_rows = []

for threshold in quantiles:
    accepted = final_dev_predictions[
        final_dev_predictions["top1_top2_margin"]
        >= threshold
    ]

    coverage = (
        len(accepted)
        / len(final_dev_predictions)
        if len(final_dev_predictions)
        else 0.0
    )

    accuracy = (
        float(accepted["top1_correct"].mean())
        if len(accepted)
        else None
    )

    selective_rows.append({
        "margin_threshold": float(threshold),
        "coverage": float(coverage),
        "top1_accuracy_given_accept":
            accuracy,
        "accepted_studies": int(len(accepted)),
    })

selective_curve_df = pd.DataFrame(
    selective_rows
)

print()
print("DEV selective curve (DESCRIPTIVE; threshold NOT frozen):")
print(selective_curve_df.to_string(index=False))

final_train_predictions.to_csv(
    METRICS_DIR / "scorer_train_predictions.csv",
    index=False,
)

final_dev_predictions.to_csv(
    METRICS_DIR / "scorer_dev_predictions.csv",
    index=False,
)

per_candidate_count.to_csv(
    METRICS_DIR / "scorer_dev_by_candidate_count.csv",
    index=False,
)

selective_curve_df.to_csv(
    METRICS_DIR / "scorer_dev_selective_curve.csv",
    index=False,
)

FINAL TRAIN:
{
  "loss": 0.1898582232504738,
  "top1_accuracy": 0.9276061776061776,
  "top2_accuracy": 0.9913127413127413,
  "mrr": 0.9617209045780475,
  "n_studies": 1036
}

FINAL DEV:
{
  "loss": 0.2190729961029351,
  "top1_accuracy": 0.9343629343629344,
  "top2_accuracy": 0.972972972972973,
  "mrr": 0.9617117117117117,
  "n_studies": 259
}

DEV by candidate count:
 n_candidates  studies  top1_accuracy  top2_accuracy  mean_margin
            2       31       1.000000       1.000000     0.999466
            3      106       0.990566       1.000000     0.923395
            4       87       0.908046       0.977011     0.769401
            5       20       0.850000       0.950000     0.569316
            6        7       0.714286       0.857143     0.349771
            7        5       1.000000       1.000000     0.233276
            8        1       0.000000       0.000000     0.545276
           10        1       0.000000       0.000000     0.040386
           14        1       0.00000

In [23]:
# ============================================================
# 17 — Save .pt checkpoint + training evidence
# ============================================================

BEST_MODEL_PATH = (
    MODEL_DIR
    / "absolute_lumbar_window_scorer_best.pt"
)

history_df = pd.DataFrame(history)
history_df.to_csv(
    METRICS_DIR / "training_history.csv",
    index=False,
)

target_qc_counts = {
    str(k): int(v)
    for k, v in
    study_df["target_qc_status"]
    .value_counts(dropna=False)
    .to_dict()
    .items()
}

runtime_counts = {
    str(k): int(v)
    for k, v in
    study_df["runtime_status"]
    .value_counts(dropna=False)
    .to_dict()
    .items()
}

checkpoint_payload = {
    "checkpoint_type":
        "post_e50_67B2_absolute_lumbar_window_scorer",

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "model_state_dict":
        {
            k: v.detach().cpu()
            for k, v
            in scorer_model.state_dict().items()
        },

    "model_config": {
        "input_dim": len(FEATURE_NAMES),
        "hidden1": 32,
        "hidden2": 16,
        "output_semantics":
            "scalar score per candidate 5-disc window",
        "study_level_decision":
            "argmax score among candidate windows",
    },

    "feature_names":
        FEATURE_NAMES,

    "feature_mean":
        feature_mean,

    "feature_std":
        feature_std,

    "seed":
        SEED,

    "source_sagittal_checkpoint_sha256":
        EXPECTED_CHECKPOINT_SHA256,

    "dataset_csv_sha256":
        EXPECTED_CSV_SHA256,

    "outer_split_contract": {
        "train": 1382,
        "validation": 296,
        "internal_test": 297,
    },

    "training_scope":
        "OUTER_TRAIN_ONLY",

    "scorer_internal_split": {
        "train_studies":
            len(train_dataset),
        "dev_studies":
            len(dev_dataset),
    },

    "target_definition":
        (
            "Among all contiguous 5-disc candidate windows, "
            "choose the window with minimum post-hoc total "
            "longitudinal Hungarian distance to the five RSNA "
            "Spinal Canal Stenosis ABSOLUTE_LEVEL_REFERENCE_POINT "
            "coordinates, only when those five TRAIN references "
            "are strictly monotonic along the GT-free DICOM-oriented "
            "SUPERIOR_TO_INFERIOR axis."
        ),

    "target_qc_counts":
        target_qc_counts,

    "runtime_counts":
        runtime_counts,

    "best_epoch":
        int(best_epoch) if best_epoch is not None else None,

    "train_metrics":
        final_train_metrics,

    "dev_metrics":
        final_dev_metrics,

    "abstention_policy": {
        "status":
            "NOT_FROZEN",
        "candidate_signal":
            "top1_probability_minus_top2_probability",
        "note":
            (
                "67B2 stores the DEV selective curve but does not "
                "choose a final threshold automatically. Freeze the "
                "threshold before external validation."
            ),
    },

    "validation_external_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "automatic_disc_localization_validated":
        False,
}

torch.save(
    checkpoint_payload,
    BEST_MODEL_PATH,
)

saved_sha = sha256_file(BEST_MODEL_PATH)

summary = {
    "model_path":
        str(BEST_MODEL_PATH),
    "model_sha256":
        saved_sha,
    "best_epoch":
        best_epoch,
    "train_metrics":
        final_train_metrics,
    "dev_metrics":
        final_dev_metrics,
    "target_qc_counts":
        target_qc_counts,
    "runtime_counts":
        runtime_counts,
    "feature_rows":
        int(len(feature_df)),
    "study_rows":
        int(len(study_df)),
    "eligible_scorer_studies":
        int(len(eligible_scorer_studies)),
    "scorer_train_studies":
        int(len(train_dataset)),
    "scorer_dev_studies":
        int(len(dev_dataset)),
    "abstention_policy_frozen":
        False,
    "external_validation_accessed":
        False,
    "internal_test_accessed":
        False,
    "official_test_accessed":
        False,
    "AUTOMATIC_DISC_LOCALIZATION_VALIDATED":
        False,
}

(METRICS_DIR / "67B2_training_summary.json").write_text(
    json.dumps(summary, indent=2, default=str),
    encoding="utf-8",
)

print("==============================================")
print("67B2 TRAINING COMPLETE")
print("==============================================")
print("Model:", BEST_MODEL_PATH)
print("SHA256:", saved_sha)
print("Best epoch:", best_epoch)
print()
print("TRAIN top1:", final_train_metrics["top1_accuracy"])
print("DEV top1:", final_dev_metrics["top1_accuracy"])
print("DEV top2:", final_dev_metrics["top2_accuracy"])
print("DEV MRR:", final_dev_metrics["mrr"])
print()
print("Abstention policy frozen: False")
print("External validation accessed: False")
print("Internal test accessed: False")
print("Official test accessed: False")
print("AUTOMATIC_DISC_LOCALIZATION_VALIDATED: False")

67B2 TRAINING COMPLETE
Model: /content/drive/MyDrive/PFI_MVP/models/research/67B2/absolute_lumbar_window_scorer_best.pt
SHA256: 527c00a81ac4df86e742cfe646bd8ffc1077d42cb30d2700d67d15e2d9ca3ab7
Best epoch: 12

TRAIN top1: 0.9276061776061776
DEV top1: 0.9343629343629344
DEV top2: 0.972972972972973
DEV MRR: 0.9617117117117117

Abstention policy frozen: False
External validation accessed: False
Internal test accessed: False
Official test accessed: False
AUTOMATIC_DISC_LOCALIZATION_VALIDATED: False


## Qué pasarme al terminar

No hace falta copiar todo el notebook. Pasame:

1. El bloque `Feature extraction complete/resumed` + los `target_qc_status`.
2. El tamaño de `Eligible scorer studies / scorer train / scorer dev`.
3. El último bloque `67B2 TRAINING COMPLETE`.
4. La tabla `DEV by candidate count`.
5. La tabla `DEV selective curve`.

Con eso decidimos rápidamente si:
- el scorer ya es suficientemente bueno;
- qué regla de abstención congelar usando **sólo DEV interno**;
- y recién entonces si hacemos la única corrida de `validation=296`.

**No abras validation manualmente antes de ese punto.**

In [24]:
# ============================================================
# 18 — FREEZE 67B2 BEFORE LOCKED VALIDATION
#
# This cell MUST run BEFORE external validation is accessed.
#
# Freezes:
# - trained scorer weights
# - feature contract
# - normalization
# - decision rule
# - abstention threshold
# - validation gates
#
# DOES NOT overwrite the trained checkpoint.
# ============================================================

import copy
import json
from datetime import datetime, timezone
from pathlib import Path

# ------------------------------------------------------------
# Safety
# ------------------------------------------------------------

assert RUN_LOCKED_VALIDATION is False
assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False

assert BEST_MODEL_PATH.is_file(), (
    f"Missing trained scorer: {BEST_MODEL_PATH}"
)

TRAINED_SCORER_SHA256 = sha256_file(
    BEST_MODEL_PATH
)

assert (
    TRAINED_SCORER_SHA256
    ==
    "527c00a81ac4df86e742cfe646bd8ffc1077d42cb30d2700d67d15e2d9ca3ab7"
), (
    "HARD STOP: trained scorer SHA mismatch."
)

print(
    "Trained scorer SHA:",
    TRAINED_SCORER_SHA256,
)


# ============================================================
# 1 — Freeze exact DEV abstention point
# ============================================================

# We already selected the DEV operating point:
# accepted studies = 194 / 259
#
# Pull the full-precision threshold directly from the
# saved in-memory selective curve rather than using the
# rounded printed value.

_selected_policy_rows = (
    selective_curve_df[
        selective_curve_df[
            "accepted_studies"
        ] == 194
    ]
)

assert len(_selected_policy_rows) == 1, (
    "Could not uniquely resolve the preselected "
    "194/259 DEV operating point."
)

_selected_policy = (
    _selected_policy_rows.iloc[0]
)

FROZEN_MARGIN_THRESHOLD = float(
    _selected_policy[
        "margin_threshold"
    ]
)

FROZEN_DEV_COVERAGE = float(
    _selected_policy[
        "coverage"
    ]
)

FROZEN_DEV_ACCURACY_GIVEN_ACCEPT = float(
    _selected_policy[
        "top1_accuracy_given_accept"
    ]
)

assert abs(
    FROZEN_MARGIN_THRESHOLD
    - 0.745862
) < 1e-5

assert int(
    _selected_policy[
        "accepted_studies"
    ]
) == 194


print(
    "Frozen margin threshold:",
    repr(
        FROZEN_MARGIN_THRESHOLD
    )
)

print(
    "DEV coverage at freeze:",
    FROZEN_DEV_COVERAGE,
)

print(
    "DEV accepted accuracy at freeze:",
    FROZEN_DEV_ACCURACY_GIVEN_ACCEPT,
)


# ============================================================
# 2 — Freeze operational inference policy
# ============================================================

FROZEN_OPERATIONAL_POLICY = {

    "direction":
        (
            "67B1 DICOM GT-free "
            "SUPERIOR_TO_INFERIOR rule"
        ),

    "n_instances_lt_5":
        "ABSTAIN",

    # No window ambiguity exists.
    "n_candidate_windows_eq_1":
        "ACCEPT_SINGLE_WINDOW",

    "n_candidate_windows_ge_2":
        "USE_67B2_SCORER",

    "scorer_decision":
        "ARGMAX_CANDIDATE_SCORE",

    "abstention_signal":
        (
            "top1_probability"
            "_minus_top2_probability"
        ),

    "abstention_threshold":
        FROZEN_MARGIN_THRESHOLD,

    "if_margin_below_threshold":
        "ABSTAIN_WINDOW",
}


# ============================================================
# 3 — PREDECLARE validation gates
#
# IMPORTANT:
# These values are frozen BEFORE validation.
#
# They are engineering validation gates, not clinical
# diagnostic thresholds.
# ============================================================

FROZEN_VALIDATION_GATES = {

    # Orientation:
    # allow isolated MIXED reference anomalies,
    # but no globally REVERSED study.
    "direction_same_rate_min":
        0.95,

    "direction_reversed_count_max":
        0,

    # Raw scorer quality among multi-window,
    # GT-evaluable validation cases.
    "raw_top1_accuracy_min":
        0.90,

    # Selective scorer behavior.
    "selective_accuracy_min":
        0.97,

    "selective_coverage_min":
        0.65,

    # Full operational absolute naming.
    "pipeline_accuracy_given_emit_min":
        0.97,

    "pipeline_gt_evaluable_coverage_min":
        0.60,
}


# ============================================================
# 4 — Create immutable PREVALIDATION checkpoint
# ============================================================

FROZEN_MODEL_PATH = (
    MODEL_DIR
    / "absolute_lumbar_window_scorer_frozen_prevalidation.pt"
)

PREVALIDATION_POLICY_PATH = (
    METRICS_DIR
    / "67B2_prevalidation_policy.json"
)


if FROZEN_MODEL_PATH.is_file():

    # Never silently overwrite a previously frozen artifact.
    frozen_checkpoint = torch.load(
        FROZEN_MODEL_PATH,
        map_location="cpu",
        weights_only=False,
    )

    assert (
        frozen_checkpoint[
            "source_trained_checkpoint_sha256"
        ]
        == TRAINED_SCORER_SHA256
    )

    assert abs(
        float(
            frozen_checkpoint[
                "abstention_policy"
            ][
                "threshold"
            ]
        )
        -
        FROZEN_MARGIN_THRESHOLD
    ) < 1e-12

    assert (
        frozen_checkpoint[
            "validation_gate_contract"
        ]
        ==
        FROZEN_VALIDATION_GATES
    )

    print(
        "Existing frozen prevalidation artifact "
        "verified; not overwritten."
    )

else:

    trained_checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location="cpu",
        weights_only=False,
    )

    frozen_checkpoint = copy.deepcopy(
        trained_checkpoint
    )

    frozen_checkpoint[
        "checkpoint_type"
    ] = (
        "post_e50_67B2_absolute_lumbar_"
        "window_scorer_FROZEN_PREVALIDATION"
    )

    frozen_checkpoint[
        "frozen_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    frozen_checkpoint[
        "source_trained_checkpoint_sha256"
    ] = TRAINED_SCORER_SHA256

    frozen_checkpoint[
        "operational_policy"
    ] = FROZEN_OPERATIONAL_POLICY

    frozen_checkpoint[
        "abstention_policy"
    ] = {

        "status":
            "FROZEN_PREVALIDATION",

        "signal":
            (
                "top1_probability"
                "_minus_top2_probability"
            ),

        "threshold":
            FROZEN_MARGIN_THRESHOLD,

        "dev_selected_operating_point":
            {

                "accepted_studies":
                    194,

                "total_dev_studies":
                    259,

                "coverage":
                    FROZEN_DEV_COVERAGE,

                "accuracy_given_accept":
                    FROZEN_DEV_ACCURACY_GIVEN_ACCEPT,
            },
    }

    frozen_checkpoint[
        "validation_gate_contract"
    ] = FROZEN_VALIDATION_GATES

    frozen_checkpoint[
        "external_validation_accessed"
    ] = False

    frozen_checkpoint[
        "internal_test_accessed"
    ] = False

    frozen_checkpoint[
        "official_test_accessed"
    ] = False

    frozen_checkpoint[
        "automatic_disc_localization_validated"
    ] = False

    torch.save(
        frozen_checkpoint,
        FROZEN_MODEL_PATH,
    )


FROZEN_MODEL_SHA256 = sha256_file(
    FROZEN_MODEL_PATH
)


# ============================================================
# 5 — Save human-readable policy manifest
# ============================================================

prevalidation_manifest = {

    "stage":
        "67B2_PREVALIDATION_FREEZE",

    "created_before_locked_validation":
        True,

    "source_trained_checkpoint":
        str(BEST_MODEL_PATH),

    "source_trained_checkpoint_sha256":
        TRAINED_SCORER_SHA256,

    "frozen_checkpoint":
        str(FROZEN_MODEL_PATH),

    "frozen_checkpoint_sha256":
        FROZEN_MODEL_SHA256,

    "best_epoch":
        12,

    "feature_count":
        len(FEATURE_NAMES),

    "operational_policy":
        FROZEN_OPERATIONAL_POLICY,

    "validation_gates":
        FROZEN_VALIDATION_GATES,

    "dev_metrics_at_training":
        final_dev_metrics,

    "validation_external_accessed":
        False,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,
}


PREVALIDATION_POLICY_PATH.write_text(
    json.dumps(
        prevalidation_manifest,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


print()
print(
    "=============================================="
)

print(
    "67B2 PREVALIDATION FREEZE COMPLETE"
)

print(
    "=============================================="
)

print(
    "Source scorer SHA:",
    TRAINED_SCORER_SHA256,
)

print(
    "Frozen scorer:",
    FROZEN_MODEL_PATH,
)

print(
    "Frozen scorer SHA:",
    FROZEN_MODEL_SHA256,
)

print(
    "Margin threshold:",
    repr(
        FROZEN_MARGIN_THRESHOLD
    ),
)

print()

print(
    "VALIDATION STILL UNOPENED: PASS"
)

print(
    "INTERNAL TEST STILL LOCKED: PASS"
)

print(
    "OFFICIAL TEST STILL LOCKED: PASS"
)

Trained scorer SHA: 527c00a81ac4df86e742cfe646bd8ffc1077d42cb30d2700d67d15e2d9ca3ab7
Frozen margin threshold: 0.7458617463707924
DEV coverage at freeze: 0.749034749034749
DEV accepted accuracy at freeze: 0.9896907216494846

67B2 PREVALIDATION FREEZE COMPLETE
Source scorer SHA: 527c00a81ac4df86e742cfe646bd8ffc1077d42cb30d2700d67d15e2d9ca3ab7
Frozen scorer: /content/drive/MyDrive/PFI_MVP/models/research/67B2/absolute_lumbar_window_scorer_frozen_prevalidation.pt
Frozen scorer SHA: 08aa8138b8ef082352a531d9395571630e41dc9ede32e5d765635c559e1aa3d3
Margin threshold: 0.7458617463707924

VALIDATION STILL UNOPENED: PASS
INTERNAL TEST STILL LOCKED: PASS
OFFICIAL TEST STILL LOCKED: PASS


In [25]:
# ============================================================
# 19 — 67B2 LOCKED VALIDATION
#
# ONE frozen evaluation of outer validation = 296 studies.
#
# Prediction is completed BEFORE GT is resolved.
#
# NO:
# - retraining
# - threshold tuning
# - feature tuning
# - model selection
# - internal_test
# - official test
#
# Resumable cache is allowed only with the EXACT SAME
# frozen checkpoint / threshold / split contract.
# ============================================================

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)

import gc
import hashlib
import json
import time


# ============================================================
# 0 — LOCKED CONTRACT
# ============================================================

assert FROZEN_MODEL_PATH.is_file()

assert (
    sha256_file(
        FROZEN_MODEL_PATH
    )
    ==
    FROZEN_MODEL_SHA256
)

assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False


LOCKED_VALIDATION_RAW = sorted(
    str(x)
    for x
    in outer_split[
        "validation"
    ]
)

assert len(
    LOCKED_VALIDATION_RAW
) == 296


validation_split_sha = hashlib.sha256(
    "\n".join(
        LOCKED_VALIDATION_RAW
    ).encode("utf-8")
).hexdigest()


LOCKED_VAL_DIR = (
    METRICS_DIR
    / "locked_validation"
)

LOCKED_VAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


LOCKED_VAL_CONTRACT_PATH = (
    LOCKED_VAL_DIR
    / "locked_validation_contract.json"
)

LOCKED_VAL_RESULTS_PATH = (
    LOCKED_VAL_DIR
    / "locked_validation_predictions.csv"
)


locked_contract = {

    "stage":
        "67B2_LOCKED_VALIDATION",

    "frozen_model_sha256":
        FROZEN_MODEL_SHA256,

    "source_trained_model_sha256":
        TRAINED_SCORER_SHA256,

    "margin_threshold":
        FROZEN_MARGIN_THRESHOLD,

    "validation_split_count":
        296,

    "validation_split_sha256":
        validation_split_sha,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "threshold_tuning_allowed":
        False,

    "model_tuning_allowed":
        False,
}


if LOCKED_VAL_CONTRACT_PATH.is_file():

    existing_contract = json.loads(
        LOCKED_VAL_CONTRACT_PATH.read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_contract
        ==
        locked_contract
    ), (
        "HARD STOP: validation contract differs "
        "from prior run."
    )

    print(
        "Resuming existing LOCKED validation "
        "with identical contract."
    )

else:

    # This is the formal point where validation is opened.
    LOCKED_VAL_CONTRACT_PATH.write_text(
        json.dumps(
            locked_contract,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "LOCKED VALIDATION CONTRACT CREATED."
    )


# ============================================================
# 1 — Load frozen scorer artifact FROM DISK
# ============================================================

locked_ckpt = torch.load(
    FROZEN_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)


locked_feature_names = list(
    locked_ckpt[
        "feature_names"
    ]
)

locked_feature_mean = np.asarray(
    locked_ckpt[
        "feature_mean"
    ],
    dtype=np.float32,
)

locked_feature_std = np.asarray(
    locked_ckpt[
        "feature_std"
    ],
    dtype=np.float32,
)


locked_model_config = (
    locked_ckpt[
        "model_config"
    ]
)


locked_scorer = (
    AbsoluteLumbarWindowScorer(

        input_dim=
            int(
                locked_model_config[
                    "input_dim"
                ]
            ),

        hidden1=
            int(
                locked_model_config[
                    "hidden1"
                ]
            ),

        hidden2=
            int(
                locked_model_config[
                    "hidden2"
                ]
            ),
    )
)


locked_scorer.load_state_dict(
    locked_ckpt[
        "model_state_dict"
    ],
    strict=True,
)


locked_scorer.to(
    DEVICE
)

locked_scorer.eval()


assert (
    locked_feature_names
    ==
    FEATURE_NAMES
)

assert len(
    locked_feature_names
) == 49


print(
    "Frozen scorer loaded from disk."
)

print(
    "Threshold:",
    repr(
        FROZEN_MARGIN_THRESHOLD
    )
)


# ============================================================
# 2 — Post-hoc direction evaluator
# ============================================================

def classify_direction_against_gt_posthoc(
    gt_level_xyz,
    oriented_axis,
    reference_point,
):

    if not all(
        lv in gt_level_xyz
        for lv in CANONICAL_LEVELS
    ):

        return "GT_INCOMPLETE"

    positions = project_points_onto_axis(

        [
            gt_level_xyz[lv]
            for lv
            in CANONICAL_LEVELS
        ],

        oriented_axis,

        reference_point,
    )


    deltas = np.diff(
        np.asarray(
            positions,
            dtype=np.float64,
        )
    )


    if np.all(
        deltas > 0
    ):
        return "SAME"


    if np.all(
        deltas < 0
    ):
        return "REVERSED"


    return "MIXED"


# ============================================================
# 3 — Cache / resume
# ============================================================

if LOCKED_VAL_RESULTS_PATH.is_file():

    locked_validation_df = (
        pd.read_csv(
            LOCKED_VAL_RESULTS_PATH
        )
    )

else:

    locked_validation_df = (
        pd.DataFrame()
    )


processed_val_opaque = set(

    locked_validation_df[
        "study_id_opaque"
    ]
    .astype(str)
    .tolist()

    if (
        len(
            locked_validation_df
        )
        and
        "study_id_opaque"
        in locked_validation_df.columns
    )

    else []
)


remaining_validation_raw = [

    sid

    for sid
    in LOCKED_VALIDATION_RAW

    if opaque_id(
        sid
    )
    not in processed_val_opaque
]


print()
print(
    "LOCKED validation total:",
    len(
        LOCKED_VALIDATION_RAW
    ),
)

print(
    "Already processed:",
    len(
        processed_val_opaque
    ),
)

print(
    "Remaining:",
    len(
        remaining_validation_raw
    ),
)


# ============================================================
# 4 — Locked prediction
# ============================================================

VAL_GROUP_SIZE = 16
VAL_IO_WORKERS = 8


t0 = time.time()


for group_start in range(
    0,
    len(
        remaining_validation_raw
    ),
    VAL_GROUP_SIZE,
):

    group_ids = (
        remaining_validation_raw[
            group_start:
            group_start
            + VAL_GROUP_SIZE
        ]
    )


    # --------------------------------------------------------
    # Load / preprocess
    # --------------------------------------------------------

    with ThreadPoolExecutor(
        max_workers=VAL_IO_WORKERS
    ) as executor:

        futures = {

            executor.submit(
                _load_and_prepare_study_once,
                sid,
            ): sid

            for sid
            in group_ids
        }


        loaded_group = []


        for future in as_completed(
            futures
        ):

            sid = futures[
                future
            ]

            try:

                loaded_group.append(
                    future.result()
                )

            except Exception as exc:

                loaded_group.append({

                    "study_id_raw":
                        str(sid),

                    "status":
                        (
                            "LOAD_ERROR:"
                            f"{type(exc).__name__}"
                        ),
                })


    loaded_map = {

        x[
            "study_id_raw"
        ]: x

        for x
        in loaded_group
    }


    loaded_group = [

        loaded_map[
            str(sid)
        ]

        for sid
        in group_ids
    ]


    # Frozen sagittal inference.
    group_results = (
        _infer_loaded_studies_ultrafast(
            loaded_group
        )
    )


    loaded_by_sid = {

        x[
            "study_id_raw"
        ]: x

        for x
        in loaded_group
    }


    group_rows = []


    for sid_raw in group_ids:

        sid = str(
            sid_raw
        )

        opaque = opaque_id(
            sid
        )

        loaded = loaded_by_sid[
            sid
        ]

        run = group_results[
            sid
        ]


        row = {

            "study_id_opaque":
                opaque,

            "runtime_status":
                run.get(
                    "status"
                ),

            "predicted_instance_count":
                run.get(
                    "predicted_instance_count"
                ),

            "direction_status":
                None,

            "direction_vs_gt":
                None,

            "n_candidate_windows":
                0,

            "scorer_status":
                None,

            "selected_window_index":
                None,

            "top1_probability":
                None,

            "top2_probability":
                None,

            "top1_top2_margin":
                None,

            "emitted_absolute_sequence":
                False,

            "decision_status":
                "ABSTAIN_RUNTIME",

            "gt_level_count":
                None,

            "target_qc_status":
                None,

            "gt_target_window_index":
                None,

            "raw_rank_of_target":
                None,

            "raw_top1_correct":
                None,

            "raw_top2_correct":
                None,

            "emitted_correct":
                None,
        }


        # ====================================================
        # PREDICTION PHASE — NO GT
        # ====================================================

        direction = None
        ordered_instances = None
        windows = []

        ranked_window_indices = []


        if (
            run.get(
                "status"
            )
            == "EXECUTED"
        ):

            (
                direction,
                ordered_instances,
            ) = orient_and_sort_instances(
                run
            )


            row[
                "direction_status"
            ] = direction.get(
                "status"
            )


            if (
                direction.get(
                    "status"
                )
                == "RESOLVED"
            ):

                windows = (
                    enumerate_candidate_windows(
                        len(
                            ordered_instances
                        )
                    )
                )


                row[
                    "n_candidate_windows"
                ] = len(
                    windows
                )


                # --------------------------------------------
                # No possible 5-disc window.
                # --------------------------------------------

                if len(
                    windows
                ) == 0:

                    row[
                        "decision_status"
                    ] = "ABSTAIN_N_LT_5"


                # --------------------------------------------
                # Exactly one possible window:
                # no ranking ambiguity.
                # --------------------------------------------

                elif len(
                    windows
                ) == 1:

                    selected_index = int(
                        windows[0][
                            "window_index"
                        ]
                    )

                    row[
                        "scorer_status"
                    ] = (
                        "SINGLE_WINDOW_"
                        "NO_SCORER_REQUIRED"
                    )

                    row[
                        "selected_window_index"
                    ] = selected_index

                    row[
                        "emitted_absolute_sequence"
                    ] = True

                    row[
                        "decision_status"
                    ] = (
                        "ACCEPT_SINGLE_WINDOW"
                    )

                    ranked_window_indices = [
                        selected_index
                    ]


                # --------------------------------------------
                # Multiple candidates:
                # frozen scorer + frozen abstention threshold.
                # --------------------------------------------

                else:

                    feature_matrix = []


                    for w in windows:

                        f = (
                            build_window_feature_row(
                                w,
                                ordered_instances,
                                direction[
                                    "oriented_axis"
                                ],
                                run[
                                    "predicted_axis_reference_point"
                                ],
                            )
                        )

                        feature_matrix.append(
                            [
                                float(
                                    f[name]
                                )
                                for name
                                in locked_feature_names
                            ]
                        )


                    x = np.asarray(
                        feature_matrix,
                        dtype=np.float32,
                    )


                    x = (
                        x
                        - locked_feature_mean
                    ) / locked_feature_std


                    x_tensor = (
                        torch.from_numpy(
                            x
                        )
                        .float()
                        .to(
                            DEVICE
                        )
                    )


                    with torch.inference_mode():

                        logits = (
                            locked_scorer(
                                x_tensor
                            )
                        )

                        probs = (
                            torch.softmax(
                                logits,
                                dim=0,
                            )
                            .detach()
                            .cpu()
                            .numpy()
                        )


                    order = np.argsort(
                        -probs
                    )


                    ranked_window_indices = [

                        int(
                            windows[
                                int(pos)
                            ][
                                "window_index"
                            ]
                        )

                        for pos
                        in order
                    ]


                    top1_pos = int(
                        order[0]
                    )

                    top2_pos = int(
                        order[1]
                    )


                    top1_prob = float(
                        probs[
                            top1_pos
                        ]
                    )

                    top2_prob = float(
                        probs[
                            top2_pos
                        ]
                    )

                    margin = (
                        top1_prob
                        - top2_prob
                    )


                    selected_index = int(
                        windows[
                            top1_pos
                        ][
                            "window_index"
                        ]
                    )


                    row[
                        "scorer_status"
                    ] = "SCORED"

                    row[
                        "selected_window_index"
                    ] = selected_index

                    row[
                        "top1_probability"
                    ] = top1_prob

                    row[
                        "top2_probability"
                    ] = top2_prob

                    row[
                        "top1_top2_margin"
                    ] = margin


                    if (
                        margin
                        >=
                        FROZEN_MARGIN_THRESHOLD
                    ):

                        row[
                            "emitted_absolute_sequence"
                        ] = True

                        row[
                            "decision_status"
                        ] = (
                            "ACCEPT_SCORER"
                        )

                    else:

                        row[
                            "decision_status"
                        ] = (
                            "ABSTAIN_LOW_MARGIN"
                        )


            else:

                row[
                    "decision_status"
                ] = (
                    "ABSTAIN_DIRECTION"
                )


        # ====================================================
        # PREDICTION IS NOW FROZEN.
        #
        # ONLY NOW resolve GT for post-hoc evaluation.
        # ====================================================

        if (
            loaded.get(
                "status"
            )
            == "LOADED"
        ):

            gt = (
                _resolve_gt_from_loaded_study(
                    sid,
                    loaded,
                )
            )

        else:

            gt = {}


        row[
            "gt_level_count"
        ] = len(
            gt
        )


        if (
            direction is not None
            and
            direction.get(
                "status"
            )
            == "RESOLVED"
            and
            len(gt) == 5
        ):

            row[
                "direction_vs_gt"
            ] = (
                classify_direction_against_gt_posthoc(
                    gt,
                    direction[
                        "oriented_axis"
                    ],
                    run[
                        "predicted_axis_reference_point"
                    ],
                )
            )


        if (
            direction is not None
            and
            direction.get(
                "status"
            )
            == "RESOLVED"
            and
            len(
                windows
            ) >= 1
        ):

            target_info = (
                gt_target_qc_and_best_window(
                    windows,
                    ordered_instances,
                    direction[
                        "oriented_axis"
                    ],
                    run[
                        "predicted_axis_reference_point"
                    ],
                    gt,
                )
            )


            row[
                "target_qc_status"
            ] = target_info[
                "status"
            ]


            if (
                target_info[
                    "status"
                ]
                == "TARGET_OK"
            ):

                target_window_index = int(
                    target_info[
                        "target_window_index"
                    ]
                )


                row[
                    "gt_target_window_index"
                ] = target_window_index


                if (
                    ranked_window_indices
                    and
                    target_window_index
                    in ranked_window_indices
                ):

                    rank = (
                        ranked_window_indices.index(
                            target_window_index
                        )
                        + 1
                    )

                    row[
                        "raw_rank_of_target"
                    ] = rank

                    row[
                        "raw_top1_correct"
                    ] = int(
                        rank == 1
                    )

                    row[
                        "raw_top2_correct"
                    ] = int(
                        rank <= 2
                    )


                if (
                    row[
                        "emitted_absolute_sequence"
                    ]
                ):

                    row[
                        "emitted_correct"
                    ] = int(
                        int(
                            row[
                                "selected_window_index"
                            ]
                        )
                        ==
                        target_window_index
                    )


        group_rows.append(
            row
        )


    # ========================================================
    # Incremental locked cache
    # ========================================================

    new_df = pd.DataFrame(
        group_rows
    )


    locked_validation_df = pd.concat(
        [
            locked_validation_df,
            new_df,
        ],
        ignore_index=True,
    )


    locked_validation_df = (
        locked_validation_df
        .drop_duplicates(
            subset=[
                "study_id_opaque"
            ],
            keep="last",
        )
    )


    locked_validation_df.to_csv(
        LOCKED_VAL_RESULTS_PATH,
        index=False,
    )


    del loaded_group
    del loaded_map
    del loaded_by_sid
    del group_results

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    elapsed = (
        time.time()
        - t0
    )


    print(
        f"Locked validation "
        f"{len(locked_validation_df)}/296 "
        f"| elapsed={elapsed/60:.1f} min"
    )


# ============================================================
# 5 — FINAL LOCKED METRICS
# ============================================================

assert len(
    locked_validation_df
) == 296


dfv = (
    locked_validation_df
    .copy()
)


# Direction evidence.
direction_eval = dfv[
    dfv[
        "direction_vs_gt"
    ].isin(
        [
            "SAME",
            "REVERSED",
            "MIXED",
        ]
    )
].copy()


direction_same = int(
    (
        direction_eval[
            "direction_vs_gt"
        ]
        == "SAME"
    ).sum()
)

direction_reversed = int(
    (
        direction_eval[
            "direction_vs_gt"
        ]
        == "REVERSED"
    ).sum()
)

direction_mixed = int(
    (
        direction_eval[
            "direction_vs_gt"
        ]
        == "MIXED"
    ).sum()
)

direction_same_rate = (
    direction_same
    /
    max(
        len(
            direction_eval
        ),
        1,
    )
)


# GT-evaluable target cases.
gt_eval = dfv[
    dfv[
        "target_qc_status"
    ]
    == "TARGET_OK"
].copy()


# Raw scorer only: multi-window cases.
raw_scorer_eval = gt_eval[
    gt_eval[
        "scorer_status"
    ]
    == "SCORED"
].copy()


raw_top1_accuracy = float(
    raw_scorer_eval[
        "raw_top1_correct"
    ].mean()
)

raw_top2_accuracy = float(
    raw_scorer_eval[
        "raw_top2_correct"
    ].mean()
)

raw_mrr = float(
    np.mean(
        1.0
        /
        raw_scorer_eval[
            "raw_rank_of_target"
        ].to_numpy(
            dtype=np.float64
        )
    )
)


# Selective multi-window scorer.
selective_accepted = (
    raw_scorer_eval[
        raw_scorer_eval[
            "emitted_absolute_sequence"
        ].astype(bool)
    ]
)


selective_coverage = (
    len(
        selective_accepted
    )
    /
    max(
        len(
            raw_scorer_eval
        ),
        1,
    )
)


selective_accuracy = float(
    selective_accepted[
        "emitted_correct"
    ].mean()
) if len(
    selective_accepted
) else None


# Full operational pipeline.
emitted_all = dfv[
    dfv[
        "emitted_absolute_sequence"
    ].astype(bool)
]


emitted_gt_eval = gt_eval[
    gt_eval[
        "emitted_absolute_sequence"
    ].astype(bool)
]


pipeline_coverage_all_296 = (
    len(
        emitted_all
    )
    / 296.0
)


pipeline_gt_evaluable_coverage = (
    len(
        emitted_gt_eval
    )
    /
    max(
        len(
            gt_eval
        ),
        1,
    )
)


pipeline_accuracy_given_emit = (
    float(
        emitted_gt_eval[
            "emitted_correct"
        ].mean()
    )
    if len(
        emitted_gt_eval
    )
    else None
)


# ============================================================
# 6 — GATES
# ============================================================

validation_gates = {

    "direction":
        bool(
            direction_same_rate
            >=
            FROZEN_VALIDATION_GATES[
                "direction_same_rate_min"
            ]
            and
            direction_reversed
            <=
            FROZEN_VALIDATION_GATES[
                "direction_reversed_count_max"
            ]
        ),

    "raw_top1":
        bool(
            raw_top1_accuracy
            >=
            FROZEN_VALIDATION_GATES[
                "raw_top1_accuracy_min"
            ]
        ),

    "selective":
        bool(
            selective_accuracy
            is not None
            and
            selective_accuracy
            >=
            FROZEN_VALIDATION_GATES[
                "selective_accuracy_min"
            ]
            and
            selective_coverage
            >=
            FROZEN_VALIDATION_GATES[
                "selective_coverage_min"
            ]
        ),

    "pipeline":
        bool(
            pipeline_accuracy_given_emit
            is not None
            and
            pipeline_accuracy_given_emit
            >=
            FROZEN_VALIDATION_GATES[
                "pipeline_accuracy_given_emit_min"
            ]
            and
            pipeline_gt_evaluable_coverage
            >=
            FROZEN_VALIDATION_GATES[
                "pipeline_gt_evaluable_coverage_min"
            ]
        ),
}


AUTOMATIC_DISC_LOCALIZATION_VALIDATED = bool(
    all(
        validation_gates.values()
    )
)


# ============================================================
# 7 — FINAL SUMMARY
# ============================================================

locked_validation_summary = {

    "stage":
        "67B2_LOCKED_VALIDATION",

    "frozen_model_sha256":
        FROZEN_MODEL_SHA256,

    "source_trained_model_sha256":
        TRAINED_SCORER_SHA256,

    "frozen_margin_threshold":
        FROZEN_MARGIN_THRESHOLD,

    "validation_total":
        296,

    "runtime_status_counts":
        {
            str(k): int(v)
            for k, v
            in dfv[
                "runtime_status"
            ]
            .value_counts(
                dropna=False
            )
            .to_dict()
            .items()
        },

    "target_qc_counts":
        {
            str(k): int(v)
            for k, v
            in dfv[
                "target_qc_status"
            ]
            .value_counts(
                dropna=False
            )
            .to_dict()
            .items()
        },

    "direction": {

        "evaluable":
            int(
                len(
                    direction_eval
                )
            ),

        "same":
            direction_same,

        "reversed":
            direction_reversed,

        "mixed":
            direction_mixed,

        "same_rate":
            direction_same_rate,
    },

    "raw_multiwindow_scorer": {

        "evaluable":
            int(
                len(
                    raw_scorer_eval
                )
            ),

        "top1_accuracy":
            raw_top1_accuracy,

        "top2_accuracy":
            raw_top2_accuracy,

        "mrr":
            raw_mrr,
    },

    "selective_multiwindow_scorer": {

        "accepted":
            int(
                len(
                    selective_accepted
                )
            ),

        "coverage":
            selective_coverage,

        "accuracy_given_accept":
            selective_accuracy,
    },

    "full_operational_pipeline": {

        "gt_evaluable":
            int(
                len(
                    gt_eval
                )
            ),

        "emitted_all_296":
            int(
                len(
                    emitted_all
                )
            ),

        "emitted_gt_evaluable":
            int(
                len(
                    emitted_gt_eval
                )
            ),

        "coverage_all_296":
            pipeline_coverage_all_296,

        "coverage_gt_evaluable":
            pipeline_gt_evaluable_coverage,

        "accuracy_given_emit":
            pipeline_accuracy_given_emit,
    },

    "predeclared_gates":
        FROZEN_VALIDATION_GATES,

    "gate_results":
        validation_gates,

    "AUTOMATIC_DISC_LOCALIZATION_VALIDATED":
        AUTOMATIC_DISC_LOCALIZATION_VALIDATED,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "post_validation_tuning_allowed":
        False,
}


LOCKED_VAL_SUMMARY_PATH = (
    LOCKED_VAL_DIR
    / "67B2_locked_validation_summary.json"
)


LOCKED_VAL_SUMMARY_PATH.write_text(
    json.dumps(
        locked_validation_summary,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


print()
print(
    "=============================================="
)

print(
    "67B2 LOCKED VALIDATION COMPLETE"
)

print(
    "=============================================="
)

print(
    "Validation total:",
    296
)

print()

print(
    "DIRECTION:"
)

print(
    "  evaluable:",
    len(
        direction_eval
    )
)

print(
    "  SAME:",
    direction_same
)

print(
    "  REVERSED:",
    direction_reversed
)

print(
    "  MIXED:",
    direction_mixed
)

print(
    "  SAME rate:",
    direction_same_rate
)

print()

print(
    "RAW MULTI-WINDOW SCORER:"
)

print(
    "  evaluable:",
    len(
        raw_scorer_eval
    )
)

print(
    "  Top-1:",
    raw_top1_accuracy
)

print(
    "  Top-2:",
    raw_top2_accuracy
)

print(
    "  MRR:",
    raw_mrr
)

print()

print(
    "SELECTIVE MULTI-WINDOW:"
)

print(
    "  accepted:",
    len(
        selective_accepted
    )
)

print(
    "  coverage:",
    selective_coverage
)

print(
    "  accuracy:",
    selective_accuracy
)

print()

print(
    "FULL PIPELINE:"
)

print(
    "  GT evaluable:",
    len(
        gt_eval
    )
)

print(
    "  emitted all 296:",
    len(
        emitted_all
    )
)

print(
    "  coverage all 296:",
    pipeline_coverage_all_296
)

print(
    "  GT-evaluable coverage:",
    pipeline_gt_evaluable_coverage
)

print(
    "  accuracy given emit:",
    pipeline_accuracy_given_emit
)

print()

print(
    "GATES:",
    validation_gates
)

print()

print(
    "AUTOMATIC_DISC_LOCALIZATION_VALIDATED:",
    AUTOMATIC_DISC_LOCALIZATION_VALIDATED
)

print()

print(
    "Internal test accessed: False"
)

print(
    "Official test accessed: False"
)

LOCKED VALIDATION CONTRACT CREATED.
Frozen scorer loaded from disk.
Threshold: 0.7458617463707924

LOCKED validation total: 296
Already processed: 0
Remaining: 296
Locked validation 16/296 | elapsed=0.7 min
Locked validation 32/296 | elapsed=1.5 min
Locked validation 48/296 | elapsed=2.4 min
Locked validation 64/296 | elapsed=3.0 min
Locked validation 80/296 | elapsed=4.0 min
Locked validation 96/296 | elapsed=4.9 min
Locked validation 112/296 | elapsed=5.4 min
Locked validation 128/296 | elapsed=6.4 min
Locked validation 144/296 | elapsed=6.9 min
Locked validation 160/296 | elapsed=7.6 min
Locked validation 176/296 | elapsed=8.4 min
Locked validation 192/296 | elapsed=9.0 min
Locked validation 208/296 | elapsed=9.7 min
Locked validation 224/296 | elapsed=10.6 min
Locked validation 240/296 | elapsed=11.4 min
Locked validation 256/296 | elapsed=12.1 min
Locked validation 272/296 | elapsed=12.9 min
Locked validation 288/296 | elapsed=14.0 min
Locked validation 296/296 | elapsed=14.3 min


In [26]:
# ============================================================
# 20 — FORMAL CLOSE 67B2 AFTER LOCKED VALIDATION
#
# Creates a NEW validated checkpoint.
# Does NOT overwrite:
# - trained scorer
# - frozen prevalidation scorer
#
# No tuning. No retraining. No internal_test.
# ============================================================

import copy
import json
from datetime import datetime, timezone


# ------------------------------------------------------------
# 0 — Hard validation checks
# ------------------------------------------------------------

assert (
    sha256_file(FROZEN_MODEL_PATH)
    ==
    "08aa8138b8ef082352a531d9395571630e41dc9ede32e5d765635c559e1aa3d3"
)

assert (
    TRAINED_SCORER_SHA256
    ==
    "527c00a81ac4df86e742cfe646bd8ffc1077d42cb30d2700d67d15e2d9ca3ab7"
)

assert locked_validation_summary[
    "validation_total"
] == 296

assert all(
    locked_validation_summary[
        "gate_results"
    ].values()
)

assert (
    locked_validation_summary[
        "AUTOMATIC_DISC_LOCALIZATION_VALIDATED"
    ]
    is True
)

assert RSNA_INTERNAL_TEST_LOCKED is True
assert OFFICIAL_TEST_ACCESSED is False


# ------------------------------------------------------------
# 1 — Freeze exact validation evidence
# ------------------------------------------------------------

LOCKED_VALIDATION_SUMMARY_SHA256 = (
    sha256_file(
        LOCKED_VAL_SUMMARY_PATH
    )
)

LOCKED_VALIDATION_RESULTS_SHA256 = (
    sha256_file(
        LOCKED_VAL_RESULTS_PATH
    )
)

print(
    "Locked validation summary SHA:",
    LOCKED_VALIDATION_SUMMARY_SHA256,
)

print(
    "Locked validation predictions SHA:",
    LOCKED_VALIDATION_RESULTS_SHA256,
)


# ------------------------------------------------------------
# 2 — Create validated artifact
# ------------------------------------------------------------

VALIDATED_MODEL_PATH = (
    MODEL_DIR
    / "absolute_lumbar_window_scorer_validated.pt"
)

FINAL_CLOSURE_PATH = (
    METRICS_DIR
    / "67B2_final_closure.json"
)


if VALIDATED_MODEL_PATH.is_file():

    validated_checkpoint = torch.load(
        VALIDATED_MODEL_PATH,
        map_location="cpu",
        weights_only=False,
    )

    assert (
        validated_checkpoint[
            "source_prevalidation_checkpoint_sha256"
        ]
        ==
        FROZEN_MODEL_SHA256
    )

    assert (
        validated_checkpoint[
            "automatic_disc_localization_validated"
        ]
        is True
    )

    print(
        "Existing validated artifact verified; "
        "not overwritten."
    )

else:

    prevalidation_checkpoint = torch.load(
        FROZEN_MODEL_PATH,
        map_location="cpu",
        weights_only=False,
    )

    validated_checkpoint = copy.deepcopy(
        prevalidation_checkpoint
    )

    validated_checkpoint[
        "checkpoint_type"
    ] = (
        "post_e50_67B2_absolute_lumbar_"
        "window_scorer_VALIDATED"
    )

    validated_checkpoint[
        "validated_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    validated_checkpoint[
        "source_prevalidation_checkpoint_sha256"
    ] = FROZEN_MODEL_SHA256

    validated_checkpoint[
        "locked_validation_summary_sha256"
    ] = LOCKED_VALIDATION_SUMMARY_SHA256

    validated_checkpoint[
        "locked_validation_predictions_sha256"
    ] = LOCKED_VALIDATION_RESULTS_SHA256

    validated_checkpoint[
        "locked_validation_metrics"
    ] = locked_validation_summary

    validated_checkpoint[
        "external_validation_accessed"
    ] = True

    validated_checkpoint[
        "internal_test_accessed"
    ] = False

    validated_checkpoint[
        "official_test_accessed"
    ] = False

    validated_checkpoint[
        "post_validation_tuning_allowed"
    ] = False

    validated_checkpoint[
        "automatic_disc_localization_validated"
    ] = True

    torch.save(
        validated_checkpoint,
        VALIDATED_MODEL_PATH,
    )


VALIDATED_MODEL_SHA256 = (
    sha256_file(
        VALIDATED_MODEL_PATH
    )
)


# ------------------------------------------------------------
# 3 — Final closure manifest
# ------------------------------------------------------------

final_closure = {

    "stage":
        "67B2_FORMALLY_CLOSED",

    "decision":
        "PASS",

    "validated_model":
        str(
            VALIDATED_MODEL_PATH
        ),

    "validated_model_sha256":
        VALIDATED_MODEL_SHA256,

    "source_trained_model_sha256":
        TRAINED_SCORER_SHA256,

    "source_prevalidation_model_sha256":
        FROZEN_MODEL_SHA256,

    "frozen_margin_threshold":
        FROZEN_MARGIN_THRESHOLD,

    "validation_total":
        296,

    "direction_same_rate":
        locked_validation_summary[
            "direction"
        ][
            "same_rate"
        ],

    "direction_reversed":
        locked_validation_summary[
            "direction"
        ][
            "reversed"
        ],

    "raw_top1_accuracy":
        locked_validation_summary[
            "raw_multiwindow_scorer"
        ][
            "top1_accuracy"
        ],

    "raw_top2_accuracy":
        locked_validation_summary[
            "raw_multiwindow_scorer"
        ][
            "top2_accuracy"
        ],

    "selective_coverage":
        locked_validation_summary[
            "selective_multiwindow_scorer"
        ][
            "coverage"
        ],

    "selective_accuracy":
        locked_validation_summary[
            "selective_multiwindow_scorer"
        ][
            "accuracy_given_accept"
        ],

    "pipeline_coverage_all_296":
        locked_validation_summary[
            "full_operational_pipeline"
        ][
            "coverage_all_296"
        ],

    "pipeline_gt_evaluable_coverage":
        locked_validation_summary[
            "full_operational_pipeline"
        ][
            "coverage_gt_evaluable"
        ],

    "pipeline_accuracy_given_emit":
        locked_validation_summary[
            "full_operational_pipeline"
        ][
            "accuracy_given_emit"
        ],

    "gate_results":
        locked_validation_summary[
            "gate_results"
        ],

    "AUTOMATIC_DISC_LOCALIZATION_VALIDATED":
        True,

    "internal_test_accessed":
        False,

    "official_test_accessed":
        False,

    "next_stage":
        "67C_AXIAL_PAIRING",
}


FINAL_CLOSURE_PATH.write_text(
    json.dumps(
        final_closure,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


print()
print(
    "=============================================="
)

print(
    "67B2 FORMALLY CLOSED — PASS"
)

print(
    "=============================================="
)

print(
    "Validated model:",
    VALIDATED_MODEL_PATH,
)

print(
    "Validated SHA256:",
    VALIDATED_MODEL_SHA256,
)

print()

print(
    "Direction SAME rate:",
    final_closure[
        "direction_same_rate"
    ],
)

print(
    "Raw Top-1:",
    final_closure[
        "raw_top1_accuracy"
    ],
)

print(
    "Selective accuracy:",
    final_closure[
        "selective_accuracy"
    ],
)

print(
    "Selective coverage:",
    final_closure[
        "selective_coverage"
    ],
)

print(
    "Pipeline accuracy:",
    final_closure[
        "pipeline_accuracy_given_emit"
    ],
)

print(
    "Pipeline coverage:",
    final_closure[
        "pipeline_gt_evaluable_coverage"
    ],
)

print()

print(
    "AUTOMATIC_DISC_LOCALIZATION_VALIDATED: True"
)

print(
    "INTERNAL TEST STILL LOCKED: PASS"
)

print(
    "OFFICIAL TEST STILL LOCKED: PASS"
)

print()

print(
    "NEXT STAGE: 67C — AXIAL PAIRING"
)

Locked validation summary SHA: 2b74c8a5b0d8ee562f6a774e136abf95225074d32c32c7646a5ca3f0fd6e8dcb
Locked validation predictions SHA: baae269fe8f0e7bc7725a44ebebbb1431600d7ce23d56b7dd82c05a0cbcc0405

67B2 FORMALLY CLOSED — PASS
Validated model: /content/drive/MyDrive/PFI_MVP/models/research/67B2/absolute_lumbar_window_scorer_validated.pt
Validated SHA256: bb9e9d812972305cbc3dc87b96b95f78b7c9de4afc8ac4e6f071b35ae6fffd2c

Direction SAME rate: 0.9930795847750865
Raw Top-1: 0.9049295774647887
Selective accuracy: 0.990909090909091
Selective coverage: 0.7746478873239436
Pipeline accuracy: 0.9909502262443439
Pipeline coverage: 0.775438596491228

AUTOMATIC_DISC_LOCALIZATION_VALIDATED: True
INTERNAL TEST STILL LOCKED: PASS
OFFICIAL TEST STILL LOCKED: PASS

NEXT STAGE: 67C — AXIAL PAIRING
